# Setup y Lectura de Datos

In [ ]:
import pyspark
from pyspark.sql import functions as sf
from pyspark.sql import Window
from pyspark.storagelevel import StorageLevel
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import scipy.stats as scistats
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from scipy.stats import norm

# CONFIGURACIÓN DEL BU
BU = 'MEX'

# Capacidades del panel
CAPACIDADES_CONTINUAS = ['Digital', 'Multicategory', 'PedidoSugerido', 'Coolers']
CAPACIDADES_BINARIAS  = ['RTM']

# Outcome del modelo
OUTCOME = 'ingreso_neto_core'

# Muestra para validación rápida
USE_MUESTRA = True
PCT_MUESTRA = 0.2


# Optimizaciones de Spark
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "104857600")

print(f"BU configurado:            {BU}")
print(f"Capacidades continuas:     {CAPACIDADES_CONTINUAS}")
print(f"Capacidades binarias:      {CAPACIDADES_BINARIAS}")
print(f"Outcome:                   {OUTCOME}")
print(f"Muestra:                   {'Sí (' + str(int(PCT_MUESTRA*100)) + '%)' if USE_MUESTRA else 'No (panel completo)'}")

In [ ]:
# LECTURA DEL PANEL DE CAPACIDADES
input_path = f"abfss://{containerName}@{storageAccountName}.dfs.core.windows.net/CTG/{BU}/PanelCapacidades/parquet/"

panel_qa = spark.read.parquet(input_path)

# Filtrar solo las capacidades que existen en este BU
capacidades_existentes_continuas = [c for c in CAPACIDADES_CONTINUAS if c in panel_qa.columns]
capacidades_existentes_binarias  = [c for c in CAPACIDADES_BINARIAS  if c in panel_qa.columns]

# Actualizar listas a las efectivamente disponibles
CAPACIDADES_CONTINUAS = capacidades_existentes_continuas
CAPACIDADES_BINARIAS  = capacidades_existentes_binarias

# LIMPIEZA DE NULLs EN ORIGEN
# Atributos categoricos del PDV: NULL → "Sin Asignar"
print("Limpieza de NULLs:")
for col in ["territorio", "canal"]:
    if col in panel_qa.columns:
        n_nulls = panel_qa.filter(sf.col(col).isNull()).count()
        if n_nulls > 0:
            panel_qa = panel_qa.withColumn(col, sf.coalesce(sf.col(col), sf.lit("Sin Asignar")))
            print(f"  {col}: {n_nulls:,} NULLs → 'Sin Asignar'")

# Capacidades: NULL → 0
for cap in CAPACIDADES_CONTINUAS + CAPACIDADES_BINARIAS:
    if cap in panel_qa.columns:
        n_nulls = panel_qa.filter(sf.col(cap).isNull()).count()
        if n_nulls > 0:
            panel_qa = panel_qa.withColumn(cap, sf.coalesce(sf.col(cap), sf.lit(0.0)))
            print(f"  {cap}: {n_nulls:,} NULLs → 0")

print(f"\nPanel de capacidades:")
print(f"  Filas:                           {panel_qa.count():,}")
print(f"  PDVs únicos:                     {panel_qa.select('id_cliente').distinct().count():,}")
print(f"  Capacidades continuas presentes: {CAPACIDADES_CONTINUAS}")
print(f"  Capacidades binarias presentes:  {CAPACIDADES_BINARIAS}")

In [ ]:
if USE_MUESTRA:
    pdvs_unicos = panel_qa.select("id_cliente").distinct()
    pdvs_sample = pdvs_unicos.sample(fraction=PCT_MUESTRA, seed=42).cache()
    pdvs_sample.count() 
    
    panel_qa = panel_qa.join(pdvs_sample, on="id_cliente", how="inner").cache()
    panel_qa.count() 
    
    n_pdvs = panel_qa.select("id_cliente").distinct().count()
    n_filas = panel_qa.count()
    print(f"✓ Muestra aplicada (estable):")
    print(f"  PDVs:            {n_pdvs:,}")
    print(f"  Filas:           {n_filas:,}")

In [ ]:
# PANEL DIAGNÓSTICO — panel balanceado por PDV

# Rango temporal por PDV
rango_pdv = panel_qa.groupBy("id_cliente").agg(sf.min("periodo").alias("primer_mes"),sf.max("periodo").alias("ultimo_mes"))

# Atributos categoricos por PDV
catalogo_pdvs = (panel_qa.groupBy("id_cliente").agg(
        sf.first("bu", ignorenulls=True).alias("bu"),
        sf.first("territorio", ignorenulls=True).alias("territorio"),
        sf.first("canal", ignorenulls=True).alias("canal"),
        sf.first("subcanal", ignorenulls=True).alias("subcanal"),
        sf.first("tamano_cliente", ignorenulls=True).alias("tamano_cliente"),))

# Generar grilla de datos completa por PDV
grilla = (rango_pdv.withColumn("periodo_array", sf.expr("sequence(primer_mes, ultimo_mes, interval 1 month)"))
    .withColumn("periodo", sf.explode("periodo_array"))
    .select("id_cliente", "periodo"))

grilla_con_atributos = grilla.join(catalogo_pdvs, on="id_cliente", how="left")

cols_atributos = ["bu", "territorio", "canal", "subcanal", "tamano_cliente"]
panel_diagnostico = grilla_con_atributos.join(panel_qa.drop(*cols_atributos),on=["id_cliente", "periodo"],how="left")

# Marcar filas que no estaban en el panel original
panel_diagnostico = panel_diagnostico.withColumn("es_gap_relleno",sf.col(OUTCOME).isNull())

# Rellenar NULLs en métricas y capacidades con 0 
cols_metricas = [
    "ingreso_neto_total", "unit_cases_total",
    "ingreso_neto_core", "unit_cases_core",
    "ingreso_neto_online_core", "ingreso_neto_offline_core",
    "unit_cases_online_core", "unit_cases_offline_core",
    "ingreso_neto_multi", "unit_cases_multi",
    "ingreso_neto_online_multi", "ingreso_neto_offline_multi",
    "unit_cases_online_multi", "unit_cases_offline_multi",
    "ingreso_neto_online", "ingreso_neto_offline",
    "unit_cases_online", "unit_cases_offline",
]

cols_a_rellenar = cols_metricas + CAPACIDADES_CONTINUAS + CAPACIDADES_BINARIAS
cols_a_rellenar = [c for c in cols_a_rellenar if c in panel_diagnostico.columns]

for col in cols_a_rellenar:
    panel_diagnostico = panel_diagnostico.withColumn(col, sf.coalesce(sf.col(col), sf.lit(0)))

panel_diagnostico = panel_diagnostico.persist(StorageLevel.MEMORY_AND_DISK)

filas_diag = panel_diagnostico.count()
filas_gap = panel_diagnostico.filter(sf.col("es_gap_relleno")).count()
pdvs_diag = panel_diagnostico.select("id_cliente").distinct().count()

print(f"  Panel DIAGNÓSTICO:")
print(f"  Filas totales:          {filas_diag:,}")
print(f"  Filas reales:           {filas_diag - filas_gap:,}")
print(f"  Filas rellenadas: {filas_gap:,} ({filas_gap/filas_diag*100:.1f}%)")
print(f"  PDVs:                   {pdvs_diag:,}")
print(f"  Meses promedio/PDV:     {filas_diag/pdvs_diag:.1f}")

In [ ]:
# PANEL MODELO
panel_modelo = panel_qa.persist(StorageLevel.MEMORY_AND_DISK)

filas_mod = panel_modelo.count()
pdvs_mod = panel_modelo.select("id_cliente").distinct().count()

print(f"  Panel MODELO:")
print(f"  Filas:                  {filas_mod:,}")
print(f"  PDVs:                   {pdvs_mod:,}")
print(f"  Meses promedio/PDV:     {filas_mod/pdvs_mod:.1f}")

# Diagnostico de Intermitencia de PDVs

In [ ]:
# Frecuencia de compra por PDV
df_int = panel_diagnostico.withColumn("compro", sf.when(sf.col(OUTCOME) > 0, 1).otherwise(0))

resumen_pdv = (df_int.groupBy("id_cliente").agg(
        sf.count("compro").alias("meses_total"),
        sf.sum("compro").alias("meses_activo"),
        sf.first("canal", ignorenulls=True).alias("canal"),
        sf.first("tamano_cliente", ignorenulls=True).alias("tamano"),)
    .withColumn("pct_activo", sf.col("meses_activo") / sf.col("meses_total")))

mediana_activos = resumen_pdv.approxQuantile("meses_activo", [0.5], 0.01)[0]
mediana_total = resumen_pdv.approxQuantile("meses_total", [0.5], 0.01)[0]
n_pdvs = resumen_pdv.count()
pdvs_completos = resumen_pdv.filter(sf.col("pct_activo") == 1).count()
pdvs_alto = resumen_pdv.filter(sf.col("pct_activo") > 0.8).count()
pdvs_bajo = resumen_pdv.filter(sf.col("pct_activo") < 0.5).count()

print("Frecuencia de compra core por PDV")
print(f"Mediana de meses activos:            {mediana_activos:.0f} de {mediana_total:.0f}")
print(f"PDVs que compran todos los meses:    {pdvs_completos:,}  ({pdvs_completos/n_pdvs*100:.1f}%)")
print(f"PDVs con >80% meses activos:         {pdvs_alto:,}  ({pdvs_alto/n_pdvs*100:.1f}%)")
print(f"PDVs con <50% meses activos:         {pdvs_bajo:,}  ({pdvs_bajo/n_pdvs*100:.1f}%)")

In [ ]:
# Intermitencia maxima por PDV
w_pdv_orden = Window.partitionBy("id_cliente").orderBy("periodo")

df_gaps = df_int.withColumn("grupo_run",sf.sum(sf.when(sf.col("compro") != sf.lag("compro", 1).over(w_pdv_orden), 1).otherwise(0)).over(w_pdv_orden))

gaps_por_pdv = (df_gaps.filter(sf.col("compro") == 0)
    .groupBy("id_cliente", "grupo_run")
    .agg(sf.count("*").alias("largo_gap"))
    .groupBy("id_cliente")
    .agg(sf.max("largo_gap").alias("max_gap"))
)

gaps_completos = (resumen_pdv.select("id_cliente").join(gaps_por_pdv, on="id_cliente", how="left").withColumn("max_gap", sf.coalesce(sf.col("max_gap"), sf.lit(0))))

print("\nDistribución de gap máximo consecutivo:")
gaps_completos.groupBy("max_gap").count().orderBy("max_gap").show(15)

In [ ]:
resumen_pdv_pd = resumen_pdv.toPandas()
gaps_pd = gaps_completos.toPandas()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Análisis de intermitencia (sobre venta core)', fontsize=13, fontweight='bold')

# Frecuencia de compra por PDV
axes[0].hist(resumen_pdv_pd['pct_activo'], bins=20, color='#4A7DB5', alpha=0.85, edgecolor='white')
axes[0].set_title('Frecuencia de compra core por PDV')
axes[0].set_xlabel('% meses con compra core')
axes[0].set_ylabel('Cantidad de PDVs')
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
mediana_pct = resumen_pdv_pd['pct_activo'].median()
axes[0].axvline(mediana_pct, color='#C0392B', linewidth=1.5,
                linestyle='--', label=f"Mediana: {mediana_pct:.0%}")
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', linestyle='--', alpha=0.3)

# Gap máximo consecutivo
gap_counts = gaps_pd['max_gap'].value_counts().sort_index().head(13)
axes[1].bar(gap_counts.index, gap_counts.values, color='#E8593C', alpha=0.85, edgecolor='white')
axes[1].set_title('Gap máximo consecutivo por PDV')
axes[1].set_xlabel('Meses consecutivos sin compra core')
axes[1].set_ylabel('Cantidad de PDVs')
for umbral, color, label in [(3, '#C0392B', 'Umbral (3)'), (4, '#E8973A', 'Alt. (4)'), (6, '#2C8A4A', 'Alt. (6)')]:
    axes[1].axvline(umbral - 0.5, color=color, linewidth=1.5, linestyle='--', label=label)
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', linestyle='--', alpha=0.3)

# Mediana frecuencia por canal
pct_activo_canal = resumen_pdv_pd.groupby('canal')['pct_activo'].median().sort_values(ascending=True)
axes[2].barh(pct_activo_canal.index, pct_activo_canal.values, color='#5BAD8F', alpha=0.85)
axes[2].set_title('Mediana frecuencia de compra por canal')
axes[2].set_xlabel('% meses activos (mediana)')
axes[2].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[2].grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

# Estándares de Exploración y Preparación de Datos

## E1. Deflactación de variables monetarias

In [ ]:
# Fuentes de datos para obtener los indices:
#   MEX → INEGI INPC, base 2018=100
#         https://www.inegi.org.mx/temas/inpc/
#   PER → BCRP / INEI, IPC Lima Metropolitana, base Dic-2021=100
#         https://estadisticas.bcrp.gob.pe/estadisticas/series/mensuales/resultados/PN38705PM/html
#   ECU → INEC, IPC Nacional, base 2014=100
#         https://www.ecuadorencifras.gob.ec/indice-de-precios-al-consumidor/
#   ARG → INDEC, IPC Nacional Nivel General, base Dic-2016=100
#         CSV: https://www.indec.gob.ar/ftp/cuadros/economia/serie_ipc_divisiones.csv

INDICES_INFLACION = {
    'MEX': {
        'fuente': 'INEGI - INPC (base 2018=100)',
        'base_periodo': '2026-03-01',
        'base_valor': 145.544,
        'data': [
            ('2021-01-01', 110.210), ('2021-02-01', 110.907), ('2021-03-01', 111.824),
            ('2021-04-01', 112.190), ('2021-05-01', 112.419), ('2021-06-01', 113.018),
            ('2021-07-01', 113.682), ('2021-08-01', 113.899), ('2021-09-01', 114.601),
            ('2021-10-01', 115.561), ('2021-11-01', 116.884), ('2021-12-01', 117.308),
            ('2022-01-01', 118.002), ('2022-02-01', 118.981), ('2022-03-01', 120.159),
            ('2022-04-01', 120.809), ('2022-05-01', 121.022), ('2022-06-01', 122.044),
            ('2022-07-01', 122.948), ('2022-08-01', 123.803), ('2022-09-01', 124.571),
            ('2022-10-01', 125.276), ('2022-11-01', 125.997), ('2022-12-01', 126.478),
            ('2023-01-01', 127.336), ('2023-02-01', 128.046), ('2023-03-01', 128.389),
            ('2023-04-01', 128.363), ('2023-05-01', 128.084), ('2023-06-01', 128.214),
            ('2023-07-01', 128.832), ('2023-08-01', 129.545), ('2023-09-01', 130.120),
            ('2023-10-01', 130.609), ('2023-11-01', 131.445), ('2023-12-01', 132.373),
            ('2024-01-01', 133.555), ('2024-02-01', 133.681), ('2024-03-01', 134.065),
            ('2024-04-01', 134.336), ('2024-05-01', 134.087), ('2024-06-01', 134.594),
            ('2024-07-01', 136.003), ('2024-08-01', 136.013), ('2024-09-01', 136.080),
            ('2024-10-01', 136.828), ('2024-11-01', 137.424), ('2024-12-01', 137.949),
            ('2025-01-01', 138.343), ('2025-02-01', 138.726), ('2025-03-01', 139.161),
            ('2025-04-01', 139.620), ('2025-05-01', 140.012), ('2025-06-01', 140.405),
            ('2025-07-01', 140.780), ('2025-08-01', 140.867), ('2025-09-01', 141.197),
            ('2025-10-01', 141.708), ('2025-11-01', 142.645), ('2025-12-01', 143.042),
            ('2026-01-01', 143.588), ('2026-02-01', 144.307), ('2026-03-01', 145.544),
        ],
    },
    'PER': {
        'fuente': 'BCRP/INEI - IPC Lima Metropolitana (base Dic-2021=100)',
        'base_periodo': '2026-03-01',
        'base_valor': 119.59,
        'data': [
            ('2021-01-01',  94.66), ('2021-02-01',  94.54), ('2021-03-01',  95.33),
            ('2021-04-01',  95.23), ('2021-05-01',  95.49), ('2021-06-01',  95.98),
            ('2021-07-01',  96.95), ('2021-08-01',  97.90), ('2021-09-01',  98.30),
            ('2021-10-01',  98.87), ('2021-11-01',  99.22), ('2021-12-01', 100.00),
            ('2022-01-01', 100.04), ('2022-02-01', 100.35), ('2022-03-01', 101.84),
            ('2022-04-01', 102.82), ('2022-05-01', 103.21), ('2022-06-01', 104.44),
            ('2022-07-01', 105.42), ('2022-08-01', 106.13), ('2022-09-01', 106.68),
            ('2022-10-01', 107.05), ('2022-11-01', 107.60), ('2022-12-01', 108.46),
            ('2023-01-01', 108.70), ('2023-02-01', 109.02), ('2023-03-01', 110.39),
            ('2023-04-01', 111.01), ('2023-05-01', 111.36), ('2023-06-01', 111.19),
            ('2023-07-01', 111.62), ('2023-08-01', 112.04), ('2023-09-01', 112.06),
            ('2023-10-01', 111.70), ('2023-11-01', 111.52), ('2023-12-01', 111.97),
            ('2024-01-01', 111.99), ('2024-02-01', 112.62), ('2024-03-01', 113.75),
            ('2024-04-01', 113.69), ('2024-05-01', 113.59), ('2024-06-01', 113.73),
            ('2024-07-01', 114.00), ('2024-08-01', 114.32), ('2024-09-01', 114.05),
            ('2024-10-01', 113.94), ('2024-11-01', 114.05), ('2024-12-01', 114.17),
            ('2025-01-01', 114.07), ('2025-02-01', 114.28), ('2025-03-01', 115.21),
            ('2025-04-01', 115.57), ('2025-05-01', 115.51), ('2025-06-01', 115.66),
            ('2025-07-01', 115.92), ('2025-08-01', 115.59), ('2025-09-01', 115.60),
            ('2025-10-01', 115.48), ('2025-11-01', 115.61), ('2025-12-01', 115.89),
            ('2026-01-01', 116.01), ('2026-02-01', 116.81), ('2026-03-01', 119.59),
        ],
    },
    'ECU': {
        'fuente': 'INEC - IPC Nacional (base 2014=100)',
        'base_periodo': '2026-03-01',
        'base_valor': 115.26,
        'data': [
            ('2021-01-01', 104.35), ('2021-02-01', 104.44), ('2021-03-01', 104.63),
            ('2021-04-01', 104.99), ('2021-05-01', 105.08), ('2021-06-01', 104.89),
            ('2021-07-01', 105.45), ('2021-08-01', 105.57), ('2021-09-01', 105.58),
            ('2021-10-01', 105.80), ('2021-11-01', 106.18), ('2021-12-01', 106.26),
            ('2022-01-01', 107.02), ('2022-02-01', 107.27), ('2022-03-01', 107.39),
            ('2022-04-01', 108.03), ('2022-05-01', 108.63), ('2022-06-01', 109.34),
            ('2022-07-01', 109.51), ('2022-08-01', 109.54), ('2022-09-01', 109.93),
            ('2022-10-01', 110.06), ('2022-11-01', 110.05), ('2022-12-01', 110.23),
            ('2023-01-01', 110.36), ('2023-02-01', 110.38), ('2023-03-01', 110.45),
            ('2023-04-01', 110.67), ('2023-05-01', 110.77), ('2023-06-01', 111.18),
            ('2023-07-01', 111.78), ('2023-08-01', 112.34), ('2023-09-01', 112.39),
            ('2023-10-01', 112.19), ('2023-11-01', 111.74), ('2023-12-01', 111.72),
            ('2024-01-01', 111.86), ('2024-02-01', 111.96), ('2024-03-01', 112.28),
            ('2024-04-01', 113.71), ('2024-05-01', 113.58), ('2024-06-01', 112.49),
            ('2024-07-01', 113.54), ('2024-08-01', 113.79), ('2024-09-01', 113.99),
            ('2024-10-01', 113.72), ('2024-11-01', 113.42), ('2024-12-01', 112.31),
            ('2025-01-01', 112.14), ('2025-02-01', 112.24), ('2025-03-01', 112.63),
            ('2025-04-01', 112.93), ('2025-05-01', 114.10), ('2025-06-01', 114.16),
            ('2025-07-01', 114.36), ('2025-08-01', 114.71), ('2025-09-01', 114.81),
            ('2025-10-01', 115.13), ('2025-11-01', 114.62), ('2025-12-01', 114.46),
            ('2026-01-01', 114.88), ('2026-02-01', 115.11), ('2026-03-01', 115.26),
        ],
    },
    'ARG': {
        'fuente': 'INDEC - IPC Nacional Nivel General (base Dic-2016=100)',
        'base_periodo': '2026-03-01',
        'base_valor': 11077.0608,
        'data': [
            ('2021-01-01',   401.5071), ('2021-02-01',   415.8595), ('2021-03-01',   435.8657),
            ('2021-04-01',   453.6503), ('2021-05-01',   468.7250), ('2021-06-01',   483.6049),
            ('2021-07-01',   498.0987), ('2021-08-01',   510.3942), ('2021-09-01',   528.4968),
            ('2021-10-01',   547.0802), ('2021-11-01',   560.9184), ('2021-12-01',   582.4575),
            ('2022-01-01',   605.0317), ('2022-02-01',   633.4341), ('2022-03-01',   676.0566),
            ('2022-04-01',   716.9399), ('2022-05-01',   753.1470), ('2022-06-01',   793.0278),
            ('2022-07-01',   851.7610), ('2022-08-01',   911.1316), ('2022-09-01',   967.3076),
            ('2022-10-01',  1028.7060), ('2022-11-01',  1079.2787), ('2022-12-01',  1134.5875),
            ('2023-01-01',  1202.9790), ('2023-02-01',  1282.7091), ('2023-03-01',  1381.1601),
            ('2023-04-01',  1497.2147), ('2023-05-01',  1613.5895), ('2023-06-01',  1709.6115),
            ('2023-07-01',  1818.0838), ('2023-08-01',  2044.2832), ('2023-09-01',  2304.9242),
            ('2023-10-01',  2496.2730), ('2023-11-01',  2816.0628), ('2023-12-01',  3533.1922),
            ('2024-01-01',  4261.5324), ('2024-02-01',  4825.7881), ('2024-03-01',  5357.0929),
            ('2024-04-01',  5830.2271), ('2024-05-01',  6073.7000), ('2024-06-01',  6351.7145),
            ('2024-07-01',  6607.7479), ('2024-08-01',  6883.4412), ('2024-09-01',  7122.2421),
            ('2024-10-01',  7313.9542), ('2024-11-01',  7491.4314), ('2024-12-01',  7694.0075),
            ('2025-01-01',  7864.1257), ('2025-02-01',  8052.9927), ('2025-03-01',  8353.3158),
            ('2025-04-01',  8585.6078), ('2025-05-01',  8714.4871), ('2025-06-01',  8855.5681),
            ('2025-07-01',  9023.9730), ('2025-08-01',  9193.2441), ('2025-09-01',  9384.0922),
            ('2025-10-01',  9603.8623), ('2025-11-01',  9841.3581), ('2025-12-01', 10121.3715),
            ('2026-01-01', 10413.0309), ('2026-02-01', 10714.6255), ('2026-03-01', 11077.0608),
        ],
    },
}

if BU not in INDICES_INFLACION:
    raise ValueError(f"BU '{BU}' no tiene tabla de inflación configurada. BUs disponibles: {list(INDICES_INFLACION.keys())}")

config_bu = INDICES_INFLACION[BU]
INPC_BASE = config_bu['base_valor']
inpc_data = config_bu['data']

print(f"Deflactando {BU}:")
print(f"  Fuente:        {config_bu['fuente']}")
print(f"  Base período:  {config_bu['base_periodo']}")
print(f"  Base valor:    {INPC_BASE}")
print(f"  Períodos:      {len(inpc_data)} ({inpc_data[0][0]} → {inpc_data[-1][0]})")

# Cálculo del factor deflactor
deflactores_pd = [(d, idx, INPC_BASE / idx) for d, idx in inpc_data]
deflactores = spark.createDataFrame(deflactores_pd,schema=["periodo_str", "inpc", "factor_deflactor"]).withColumn("periodo", sf.to_date("periodo_str", "yyyy-MM-dd")).drop("periodo_str")

# Aplicar deflactor a panel_modelo
panel_modelo = panel_modelo.join(sf.broadcast(deflactores.select("periodo", "factor_deflactor")),on="periodo",how="left")

# Validar cobertura
sin_factor = panel_modelo.filter(sf.col("factor_deflactor").isNull()).count()
if sin_factor > 0:
    meses_sin = (panel_modelo.filter(sf.col("factor_deflactor").isNull()).select("periodo").distinct().toPandas()['periodo'].tolist())
    raise ValueError(
        f"   {sin_factor:,} filas sin factor deflactor.\n"
        f"   Meses faltantes en INDICES_INFLACION['{BU}']: {sorted(meses_sin)}"
    )
else:
    print(f"Todos los períodos tienen factor deflactor")

# Deflactar las métricas monetarias presentes
cols_a_deflactar = [c for c in ["ingreso_neto_core", "ingreso_neto_total", "ingreso_neto_multi"]
                    if c in panel_modelo.columns]

for col in cols_a_deflactar:
    panel_modelo = panel_modelo.withColumn(f"{col}_real",sf.col(col) * sf.col("factor_deflactor"))

panel_modelo = panel_modelo.persist(StorageLevel.MEMORY_AND_DISK)
panel_modelo.count()

print(f"\nPanel modelo deflactado")
print(f"  Filas:                {panel_modelo.count():,}")
print(f"  Columnas creadas:     {[f'{c}_real' for c in cols_a_deflactar]}")

In [ ]:
# Revenue CORE promedio original vs deflactado por mes
resumen_e1 = (panel_modelo.filter(sf.col("ingreso_neto_core") > 0).groupBy("periodo").agg(
        sf.avg("ingreso_neto_core").alias("ingreso_nominal"),
        sf.avg("ingreso_neto_core_real").alias("ingreso_real")
    )
    .orderBy("periodo")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(resumen_e1['periodo'], resumen_e1['ingreso_nominal'],
        color='#E8593C', linewidth=1.5, label='Revenue core original')
ax.plot(resumen_e1['periodo'], resumen_e1['ingreso_real'],
        color='#2C5F8A', linewidth=1.5, label='Revenue core deflactado')
ax.fill_between(resumen_e1['periodo'],
                resumen_e1['ingreso_real'],
                resumen_e1['ingreso_nominal'],
                alpha=0.15, color='#E8593C', label='Efecto inflación')
ax.set_title('E1: Revenue core original vs deflactado', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Revenue core promedio por PDV (meses con compra)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## E2. Tratamiento de meses con venta cero

In [ ]:
# Lógica aplicada:
# 1. Detectar PDVs con 3+ compras con ceros consecutivas
# 2. Verificar si volvieron a comprar core después de la 3era compra con cero
# 3. Si no volvieron a comprar se toma como churn definitivo, se limita el panel a partir del mes 3
# 4. Aplicar el corte al panel_modelo
# 5. Filtrar todos los ceros restantes

df_e2 = panel_diagnostico.withColumn("es_cero", sf.when(sf.col("ingreso_neto_core") == 0, 1).otherwise(0))

w_orden = Window.partitionBy("id_cliente").orderBy("periodo")

df_e2 = df_e2.withColumn("grupo_run",sf.sum(sf.when(sf.col("es_cero") != sf.lag("es_cero", 1).over(w_orden), 1).otherwise(0)).over(w_orden))
w_grupo = Window.partitionBy("id_cliente", "grupo_run").orderBy("periodo")
df_e2 = df_e2.withColumn("zeros_consec",sf.when(sf.col("es_cero") == 1, sf.row_number().over(w_grupo)).otherwise(sf.lit(0)))
tercer_cero = (df_e2.filter(sf.col("zeros_consec") == 3).groupBy("id_cliente").agg(sf.min("periodo").alias("fecha_tercer_cero")))

ultima_compra = (df_e2
    .filter(sf.col("ingreso_neto_core") > 0)
    .groupBy("id_cliente")
    .agg(sf.max("periodo").alias("ultima_compra"))
)

# Churn definitivo
cortes = (tercer_cero.join(ultima_compra, on="id_cliente", how="left")
    .filter(sf.col("ultima_compra") <= sf.col("fecha_tercer_cero"))
    .withColumn("fecha_exclusion", sf.add_months(sf.col("fecha_tercer_cero"), 1))
    .select("id_cliente", "fecha_exclusion")
).cache()

total_pdvs = panel_diagnostico.select("id_cliente").distinct().count()
pdvs_cortados = cortes.count()

print(f"E2 - Detección de churn definitivo")
print(f"Total PDVs en panel:                     {total_pdvs:,}")
print(f"PDVs con churn definitivo (cortados):    {pdvs_cortados:,}  ({pdvs_cortados/total_pdvs*100:.1f}%)")
print(f"PDVs intactos (incl. intermitentes):     {total_pdvs-pdvs_cortados:,}  ({(total_pdvs-pdvs_cortados)/total_pdvs*100:.1f}%)")

In [ ]:
filas_pre_corte = panel_modelo.count()

panel_modelo = (panel_modelo.join(cortes, on="id_cliente", how="left").filter(sf.col("fecha_exclusion").isNull() | (sf.col("periodo") < sf.col("fecha_exclusion"))).drop("fecha_exclusion"))

filas_post_corte = panel_modelo.count()

print(f"E2 - Aplicación del corte de churn")
print(f"Filas antes del corte:        {filas_pre_corte:,}")
print(f"Filas después del corte:      {filas_post_corte:,}")
print(f"Filas removidas (post-churn): {filas_pre_corte-filas_post_corte:,}  ({(filas_pre_corte-filas_post_corte)/filas_pre_corte*100:.1f}%)")

In [ ]:
filas_pre_filtro = panel_modelo.count()

panel_modelo = panel_modelo.filter(sf.col("ingreso_neto_core") > 0)

panel_modelo = panel_modelo.persist(StorageLevel.MEMORY_AND_DISK)
filas_post_filtro = panel_modelo.count()
pdvs_post = panel_modelo.select("id_cliente").distinct().count()

print(f"E2 - Filtrado de compras ceros intermitentes")
print(f"Filas antes del filtro:       {filas_pre_filtro:,}")
print(f"Filas después del filtro:     {filas_post_filtro:,}")
print(f"Ceros eliminados:             {filas_pre_filtro-filas_post_filtro:,}  ({(filas_pre_filtro-filas_post_filtro)/filas_pre_filtro*100:.1f}%)")
print(f"\nPANEL MODELO")
print(f"Filas:                        {filas_post_filtro:,}")
print(f"PDVs únicos:                  {pdvs_post:,}")
print(f"Meses promedio por PDV:       {filas_post_filtro/pdvs_post:.1f}")

### E3. Tratamiento de valores atípicos (outliers)

In [ ]:
 # Lógica: Excluimos PDVs cuyo INGRESO CORE PROMEDIO está fuera del rango P1-P99. Esto nos indica que el PDV es estructuralmente atípico, no un PDV normal con un mes de compra extraño.

# Calcular P1 y P99 sobre todas las observaciones
p01, p99 = panel_modelo.approxQuantile("ingreso_neto_core_real", [0.01, 0.99], 0.001)
print(f"P01: ${p01:,.2f}")
print(f"P99: ${p99:,.2f}")

# Calcular promedio por PDV
promedio_pdv = (panel_modelo.groupBy("id_cliente").agg(sf.avg("ingreso_neto_core_real").alias("ingreso_promedio")))

# Identificar PDVs con promedio en cola
pdvs_excluir = (promedio_pdv.filter((sf.col("ingreso_promedio") < p01) | (sf.col("ingreso_promedio") > p99)).select("id_cliente")).cache()

n_excluir = pdvs_excluir.count()
total_pdvs = panel_modelo.select("id_cliente").distinct().count()

# Desglose por cola
n_cola_inferior = (promedio_pdv.filter(sf.col("ingreso_promedio") < p01).count())
n_cola_superior = (promedio_pdv.filter(sf.col("ingreso_promedio") > p99).count())

print(f"\nPDVs a excluir (promedio fuera de [P1, P99]):  {n_excluir:,}  ({n_excluir/total_pdvs*100:.2f}%)")
print(f"  Promedio < P01:  {n_cola_inferior:,}")
print(f"  Promedio > P99:  {n_cola_superior:,}")

# Eliminar esos PDVs del panel_modelo
filas_pre = panel_modelo.count()
panel_modelo = panel_modelo.join(pdvs_excluir, on="id_cliente", how="left_anti")

panel_modelo = panel_modelo.persist(StorageLevel.MEMORY_AND_DISK)
filas_post = panel_modelo.count()
pdvs_post = panel_modelo.select("id_cliente").distinct().count()

print(f"Filas antes:                   {filas_pre:,}")
print(f"Filas después:                 {filas_post:,}")
print(f"Filas eliminadas:              {filas_pre-filas_post:,}  ({(filas_pre-filas_post)/filas_pre*100:.1f}%)")
print(f"PDVs en panel_modelo:          {pdvs_post:,}")
print(f"\nNuevo rango ingreso_neto_core_real:")
panel_modelo.select(
    sf.min("ingreso_neto_core_real").alias("min"),
    sf.expr("percentile_approx(ingreso_neto_core_real, 0.5)").alias("mediana"),
    sf.avg("ingreso_neto_core_real").alias("media"),
    sf.max("ingreso_neto_core_real").alias("max"),
).show()

In [ ]:
# Distribución de promedios por PDV
promedio_pdv_pd = promedio_pdv.toPandas()

fig, ax = plt.subplots(figsize=(14, 5))
ax.hist(promedio_pdv_pd['ingreso_promedio'].clip(upper=promedio_pdv_pd['ingreso_promedio'].quantile(0.99)),
        bins=100, color='#2C5F8A', alpha=0.7)
ax.axvline(p01, color='#C0392B', linewidth=1.5, linestyle='--', label=f'P01: ${p01:,.0f}')
ax.axvline(p99, color='#C0392B', linewidth=1.5, linestyle='-',  label=f'P99: ${p99:,.0f}')
ax.set_title('E3: Distribución del ingreso CORE PROMEDIO por PDV\n(PDVs fuera del rango se excluyen completos)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Ingreso core promedio por PDV')
ax.set_ylabel('Cantidad de PDVs')
ax.set_yscale('log')
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## E4. Análisis exploratorio previo para capacidades binarias con adopción escalonada

In [ ]:
# Universo de PDVs que panel_modelo, pero con panel balanceado
panel_e4 = panel_diagnostico.join(pdvs_excluir, on="id_cliente", how="left_anti")

panel_e4 = (panel_e4.join(cortes, on="id_cliente", how="left").filter(sf.col("fecha_exclusion").isNull() | (sf.col("periodo") < sf.col("fecha_exclusion"))).drop("fecha_exclusion"))
panel_e4 = panel_e4.join(sf.broadcast(deflactores.select("periodo", "factor_deflactor")),on="periodo",how="left")
panel_e4 = panel_e4.withColumn("ingreso_neto_core_real",sf.col("ingreso_neto_core") * sf.col("factor_deflactor"))

panel_e4 = panel_e4.persist(StorageLevel.MEMORY_AND_DISK)

filas_e4 = panel_e4.count()
pdvs_e4 = panel_e4.select("id_cliente").distinct().count()
ceros_e4 = panel_e4.filter(sf.col("ingreso_neto_core") == 0).count()

print(f"  Panel para E4 balanceado:")
print(f"  Filas:            {filas_e4:,}")
print(f"  PDVs:             {pdvs_e4:,}")
print(f"  Filas con ceros:  {ceros_e4:,} ({ceros_e4/filas_e4*100:.2f}%)")

In [ ]:
OUTCOME_E4 = "ingreso_neto_core_real"

for cap in CAPACIDADES_BINARIAS:
    print(f"E4 — {cap}")
    
    # Fecha de primera activación por PDV
    fecha_activacion = (panel_e4
        .filter(sf.col(cap) == 1)
        .groupBy("id_cliente")
        .agg(sf.min("periodo").alias("fecha_activacion"))
    )
    
    pdvs_tratados = fecha_activacion.count()
    total_pdvs_cap = panel_e4.select("id_cliente").distinct().count()
    pdvs_never = total_pdvs_cap - pdvs_tratados
    
    print(f"PDVs tratados:       {pdvs_tratados:,}  ({pdvs_tratados/total_pdvs_cap*100:.1f}%)")
    print(f"PDVs never-treated:  {pdvs_never:,}  ({pdvs_never/total_pdvs_cap*100:.1f}%)")
    
    # Curva de adopción acumulada
    cohortes_mes = (fecha_activacion.groupBy("fecha_activacion").count().orderBy("fecha_activacion").toPandas())
    cohortes_mes['pct_acumulado'] = cohortes_mes['count'].cumsum() / pdvs_tratados * 100
    
    primera = cohortes_mes['fecha_activacion'].min()
    ultima = cohortes_mes['fecha_activacion'].max()
    meses_adopcion = (ultima.year - primera.year) * 12 + (ultima.month - primera.month)
    
    if meses_adopcion <= 3:
        clasificacion = f'Concentrada ({meses_adopcion} meses) — riesgo bajo'
        color_alert = '#2C8A4A'
    elif meses_adopcion <= 12:
        clasificacion = f'Moderada ({meses_adopcion} meses) — monitorear'
        color_alert = '#E8973A'
    else:
        clasificacion = f'Dispersa ({meses_adopcion} meses) — se requiere el diagnóstico completo'
        color_alert = '#C0392B'
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
    fig.suptitle(f'E4. Curva de adopción acumulada — {cap}', fontsize=13, fontweight='bold')
    
    ax1.bar(cohortes_mes['fecha_activacion'], cohortes_mes['count'], 
            color='#4A7DB5', alpha=0.8, width=20)
    ax1.set_ylabel('PDVs nuevos por cohorte')
    ax1.grid(axis='y', linestyle='--', alpha=0.3)
    
    ax2.plot(cohortes_mes['fecha_activacion'], cohortes_mes['pct_acumulado'], 
             color='#4A7DB5', linewidth=2, marker='o', markersize=4)
    ax2.fill_between(cohortes_mes['fecha_activacion'], cohortes_mes['pct_acumulado'], 
                     alpha=0.15, color='#4A7DB5')
    ax2.axhline(50, color='gray', linestyle='--', alpha=0.5)
    ax2.axhline(90, color='gray', linestyle='--', alpha=0.5)
    ax2.set_ylabel('% acumulado de PDVs tratados')
    ax2.set_ylim(0, 105)
    ax2.grid(axis='y', linestyle='--', alpha=0.3)
    
    fig.text(0.5, -0.02,
        f"Primera cohorte: {primera.strftime('%b-%Y')}  |  "
        f"Última cohorte: {ultima.strftime('%b-%Y')}  |  "
        f"Período de adopción: {meses_adopcion} meses\nClasificación: {clasificacion}",
        ha='center', fontsize=10, color=color_alert, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#F9F9F9', alpha=0.8))
    plt.tight_layout()
    plt.show()
    
    print(f"\nClasificación: {clasificacion}")
    
    # Tamaño del grupo never-treated
    pct_never = pdvs_never / total_pdvs_cap * 100
    if pct_never >= 30:
        alerta_never = f'Never-treated grande ({pct_never:.1f}%) — riesgo acotado'
    elif pct_never >= 10:
        alerta_never = f'Never-treated moderado ({pct_never:.1f}%) — monitorear'
    else:
        alerta_never = f'Never-treated pequeño ({pct_never:.1f}%) — validar'
    
    print(f"{alerta_never}")

In [ ]:
# Revenue promedio antes y después de que cada PDV adoptó la capacidad, pero restringido a PDVs con ventana completa

VENTANA_PRE = 6   
VENTANA_POST = 6  
for cap in CAPACIDADES_BINARIAS:
    print(f"\nE4. {cap} (ventana balanceada -{VENTANA_PRE} a +{VENTANA_POST})")
    
    # Fecha de activación por PDV
    fecha_activacion = (panel_e4
        .filter(sf.col(cap) == 1)
        .groupBy("id_cliente")
        .agg(sf.min("periodo").alias("fecha_activacion"))
    )
    
    # PDVs cuya ventana completa cabe en el panel
    panel_min = panel_e4.agg(sf.min("periodo")).collect()[0][0]
    panel_max = panel_e4.agg(sf.max("periodo")).collect()[0][0]
    
    pdvs_ventana = (fecha_activacion.withColumn("fecha_min_necesaria", sf.add_months("fecha_activacion", -VENTANA_PRE))
        .withColumn("fecha_max_necesaria", sf.add_months("fecha_activacion", VENTANA_POST))
        .filter(
            (sf.col("fecha_min_necesaria") >= sf.lit(panel_min)) &
            (sf.col("fecha_max_necesaria") <= sf.lit(panel_max))
        )
        .select("id_cliente", "fecha_activacion")
    )
    
    n_balanced = pdvs_ventana.count()
    n_total = fecha_activacion.count()
    print(f"PDVs con ventana completa: {n_balanced:,} de {n_total:,} ({n_balanced/n_total*100:.1f}%)")
    
    df_tratados_bal = (panel_e4.join(pdvs_ventana, on="id_cliente", how="inner").withColumn("meses_desde_adopcion",
            ((sf.year("periodo") - sf.year("fecha_activacion")) * 12 +
             (sf.month("periodo") - sf.month("fecha_activacion"))).cast("int")
        )
        .filter((sf.col("meses_desde_adopcion") >= -VENTANA_PRE) & 
                (sf.col("meses_desde_adopcion") <= VENTANA_POST))
    )
    
    revenue_por_mes = (df_tratados_bal.groupBy("meses_desde_adopcion").agg(
            sf.avg(OUTCOME_E4).alias("revenue_promedio"),
            sf.countDistinct("id_cliente").alias("n_pdvs"),
        )
        .orderBy("meses_desde_adopcion")
        .toPandas()
    )
    
    print("\nPDVs por mes relativo (debería ser constante si la ventana es balanceada):")
    print(revenue_por_mes.to_string(index=False))
    
    # Evaluar patrón
    post = revenue_por_mes[revenue_por_mes['meses_desde_adopcion'] > 0]
    avg_0_3 = post[post['meses_desde_adopcion'] <= 3]['revenue_promedio'].mean()
    avg_4_6 = post[post['meses_desde_adopcion'] > 3]['revenue_promedio'].mean()
    
    if avg_4_6 > avg_0_3 * 1.05:
        patron = 'Efecto dinámico — sigue creciendo más allá del mes 3'
        efecto_dinamico = True
    else:
        patron = 'Efecto estático — se estabiliza en los primeros meses'
        efecto_dinamico = False
    
    fig, ax = plt.subplots(figsize=(14, 5))
    fig.suptitle(f'E4: Revenue por meses desde adopción — {cap}',
                 fontsize=13, fontweight='bold')
    ax.plot(revenue_por_mes['meses_desde_adopcion'], 
            revenue_por_mes['revenue_promedio'], 
            color='#4A7DB5', linewidth=2, marker='o', markersize=5)
    ax.axvline(0, color='#C0392B', linewidth=1.5, linestyle='--', alpha=0.8)
    ax.axvline(3, color='gray', linewidth=1, linestyle=':', alpha=0.6)
    ax.axvspan(-VENTANA_PRE, 0, alpha=0.07, color='gray', label='Pre-tratamiento')
    ax.axvspan(0, VENTANA_POST, alpha=0.07, color='#4A7DB5', label='Post-tratamiento')
    ax.set_xlabel('Meses desde adopción')
    ax.set_ylabel(f'{OUTCOME_E4} (promedio)')
    ax.set_title(f'Patrón: {patron}  |  n PDVs: {n_balanced:,}', fontsize=10, 
                 color='#C0392B' if efecto_dinamico else '#2C8A4A')
    ax.legend(fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\nE4. {patron}")

In [ ]:
# Revenue promedio por mes calendario: tratados vs never-treated

# Detectar cohorte más grande 
cohorte_sizes = (fecha_activacion.groupBy("fecha_activacion").agg(sf.count("id_cliente").alias("n_pdvs")).orderBy(sf.desc("n_pdvs")))
top = cohorte_sizes.first()
cohorte_principal = pd.to_datetime(top["fecha_activacion"])
n_cohorte = top["n_pdvs"]
cohorte_label = cohorte_principal.strftime("%b-%Y")
print(f"Cohorte más grande: {cohorte_label} ({n_cohorte:,} PDVs)")

# PDVs adoptantes de esta cohorte
pdvs_cohorte = (fecha_activacion.filter(sf.col("fecha_activacion") == cohorte_principal).select("id_cliente"))
print(f"PDVs en cohorte {cohorte_label}: {pdvs_cohorte.count():,}")

# Never-treated
pdvs_never = (panel_e4.select("id_cliente").distinct().join(fecha_activacion, on="id_cliente", how="left_anti"))
print(f"PDVs never-treated: {pdvs_never.count():,}")

# Revenue promedio por mes calendario de cada grupo
tratados_por_mes = (panel_e4.join(pdvs_cohorte, on="id_cliente", how="inner").groupBy("periodo").agg(sf.avg(OUTCOME_E4).alias("revenue_tratados")))

never_por_mes = (panel_e4.join(pdvs_never, on="id_cliente", how="inner").groupBy("periodo").agg(sf.avg(OUTCOME_E4).alias("revenue_never")))

comparacion = (tratados_por_mes.join(never_por_mes, on="periodo", how="inner").orderBy("periodo").toPandas())

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(comparacion['periodo'], comparacion['revenue_tratados'], 
        label=f'Cohorte {cohorte_label} (tratados)', color='#C0392B', linewidth=2, marker='o')
ax.plot(comparacion['periodo'], comparacion['revenue_never'], 
        label='Never-treated', color='#4A7DB5', linewidth=2, marker='s')
ax.axvline(cohorte_principal, color='gray', linewidth=1.5, linestyle='--', 
           label=f'Adopción {cohorte_label}')
ax.set_title(f'Comparación estacional: cohorte {cohorte_label} vs never-treated', 
             fontsize=13, fontweight='bold')
ax.set_ylabel(OUTCOME_E4)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Tendencias pre-adopción por cohorte
for cap in CAPACIDADES_BINARIAS:
    print(f"\nE4. {cap}")
    
    # Fecha de activación por PDV
    fecha_activacion = (panel_e4.filter(sf.col(cap) == 1).groupBy("id_cliente").agg(sf.min("periodo").alias("fecha_activacion")))
    
    # Tamaño de cada cohorte
    cohortes_tamano = (fecha_activacion.groupBy("fecha_activacion").count().orderBy(sf.col("count").desc()).toPandas())
    
    # Top 2 cohortes por tamaño
    top2 = cohortes_tamano.head(2)['fecha_activacion'].tolist()
    
    fechas_ordenadas = sorted(cohortes_tamano['fecha_activacion'].tolist())
    mediana_fecha = fechas_ordenadas[len(fechas_ordenadas) // 2]    
    
    def clasificar_cohorte_udf(fecha):
        if fecha is None:
            return None
        if fecha == top2[0]:
            return f"{pd.Timestamp(top2[0]).strftime('%b-%Y')} (top)"
        elif len(top2) > 1 and fecha == top2[1]:
            return f"{pd.Timestamp(top2[1]).strftime('%b-%Y')} (top)"
        elif fecha < mediana_fecha:
            return "Resto tempranas"
        else:
            return "Resto tardías"
    
    clasificar_udf = sf.udf(clasificar_cohorte_udf, sf.StringType())
    
    df_tratados = (panel_e4.join(fecha_activacion, on="id_cliente", how="inner")
        .withColumn("meses_desde_adopcion",
            ((sf.year("periodo") - sf.year("fecha_activacion")) * 12 +
             (sf.month("periodo") - sf.month("fecha_activacion"))).cast("int")
        )
        .withColumn("grupo_cohorte", clasificar_udf("fecha_activacion"))
    )
    
    pre = (df_tratados.filter((sf.col("meses_desde_adopcion") >= -12) & (sf.col("meses_desde_adopcion") < 0)))
    
    revenue_pre = (pre.groupBy("grupo_cohorte", "meses_desde_adopcion").agg(sf.avg(OUTCOME_E4).alias("revenue_promedio")).orderBy("grupo_cohorte", "meses_desde_adopcion").toPandas())
    
    fig, ax = plt.subplots(figsize=(14, 5))
    fig.suptitle(f'E4. Tendencias pre-adopción por cohorte — {cap}',
                 fontsize=13, fontweight='bold')
    
    colores_cohorte = ['#4A7DB5', '#E8593C', '#5BAD8F', '#E8973A']
    grupos_unicos = revenue_pre['grupo_cohorte'].unique()
    
    for grupo, color in zip(grupos_unicos, colores_cohorte):
        datos = revenue_pre[revenue_pre['grupo_cohorte'] == grupo]
        if len(datos) > 0:
            ax.plot(datos['meses_desde_adopcion'], datos['revenue_promedio'],
                    color=color, linewidth=2, marker='o', markersize=4, label=grupo)
    
    ax.axvline(0, color='gray', linewidth=1, linestyle='--', alpha=0.6)
    ax.set_xlabel('Meses antes de la adopción')
    ax.set_ylabel(f'{OUTCOME_E4} (promedio)')
    ax.legend(fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"E4. Completado, se debe revisar si las cohortes tienen trayectorias similares pre-adopción")

In [ ]:
# Revenue post-adopción por cohorte
for cap in CAPACIDADES_BINARIAS:
    print(f"\nE4. {cap} ")
    
    fecha_activacion = (panel_e4.filter(sf.col(cap) == 1).groupBy("id_cliente").agg(sf.min("periodo").alias("fecha_activacion")))
    
    cohortes_tamano = (fecha_activacion.groupBy("fecha_activacion").count().orderBy(sf.col("count").desc()).toPandas())
    
    top2 = cohortes_tamano.head(2)['fecha_activacion'].tolist()
    fechas_ordenadas = sorted(cohortes_tamano['fecha_activacion'].tolist())
    mediana_fecha = fechas_ordenadas[len(fechas_ordenadas) // 2]
    
    def clasificar_cohorte_udf(fecha):
        if fecha is None:
            return None
        if fecha == top2[0]:
            return f"{pd.Timestamp(top2[0]).strftime('%b-%Y')} (top)"
        elif len(top2) > 1 and fecha == top2[1]:
            return f"{pd.Timestamp(top2[1]).strftime('%b-%Y')} (top)"
        elif fecha < mediana_fecha:
            return "Resto tempranas"
        else:
            return "Resto tardías"
    
    clasificar_udf = sf.udf(clasificar_cohorte_udf, sf.StringType())
    
    df_tratados = (panel_e4.join(fecha_activacion, on="id_cliente", how="inner")
        .withColumn("meses_desde_adopcion",
            ((sf.year("periodo") - sf.year("fecha_activacion")) * 12 +
             (sf.month("periodo") - sf.month("fecha_activacion"))).cast("int")
        ).withColumn("grupo_cohorte", clasificar_udf("fecha_activacion")))
    
    post = (df_tratados.filter((sf.col("meses_desde_adopcion") >= 0) & (sf.col("meses_desde_adopcion") <= 12)))
    
    revenue_post = (post.groupBy("grupo_cohorte", "meses_desde_adopcion")
        .agg(sf.avg(OUTCOME_E4).alias("revenue_promedio"))
        .orderBy("grupo_cohorte", "meses_desde_adopcion")
        .toPandas()
    )
    
    fig, ax = plt.subplots(figsize=(14, 5))
    fig.suptitle(f'E4. Revenue post-adopción por cohorte — {cap}',
                 fontsize=13, fontweight='bold')
    
    colores_cohorte = ['#4A7DB5', '#E8593C', '#5BAD8F', '#E8973A']
    grupos_unicos = revenue_post['grupo_cohorte'].unique()
    
    for grupo, color in zip(grupos_unicos, colores_cohorte):
        datos = revenue_post[revenue_post['grupo_cohorte'] == grupo]
        if len(datos) > 0:
            ax.plot(datos['meses_desde_adopcion'], datos['revenue_promedio'],
                    color=color, linewidth=2, marker='o', markersize=4, label=grupo)
    
    ax.axvline(0, color='gray', linewidth=1, linestyle='--', alpha=0.6)
    ax.set_xlabel('Meses desde adopción')
    ax.set_ylabel(f'{OUTCOME_E4} (promedio)')
    ax.legend(fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    
    revenue_t0_6 = (post
        .filter(sf.col("meses_desde_adopcion") <= 6)
        .groupBy("grupo_cohorte")
        .agg(sf.avg(OUTCOME_E4).alias("revenue_prom"))
        .toPandas()
    )
    
    if len(revenue_t0_6) >= 2:
        max_r = revenue_t0_6['revenue_prom'].max()
        min_r = revenue_t0_6['revenue_prom'].min()
        diff_pct = (max_r - min_r) / min_r * 100
        color_het = '#C0392B' if diff_pct > 50 else '#2C8A4A'
        alerta_het = (f'Diferencia entre cohortes: {diff_pct:.1f}% '
                      f'{"Reportar ATT por cohorte " if diff_pct > 50 else "— homogéneo"}')
        ax.set_title(alerta_het, fontsize=10, color=color_het)
        print(f"E4. {alerta_het}")
    
    plt.tight_layout()
    plt.show()

# Criterios para definición de las características de las capacidades al momento de la implementación

## C5. Criterio para definición de tipo de medición

In [ ]:
# Criterios del marco metodológico:
#   - CV >= 0.30  
#   - IQR >= 10% de la media
# Lógica: clasificar como continua si CUALQUIERA de los dos se cumple

print("C5: Estadísticos de clasificación (capacidades continuas)")
resumen_c5 = []

for cap in CAPACIDADES_CONTINUAS:
    if cap not in panel_modelo.columns:
        print(f"\n⚠️  '{cap}' no encontrada — omitiendo.")
        continue

    n_total = panel_modelo.count()
    n_nulls = panel_modelo.filter(sf.col(cap).isNull()).count()
    n_zeros = panel_modelo.filter(sf.col(cap) == 0).count()
    n_pos = panel_modelo.filter(sf.col(cap) > 0).count()

    # Estadísticos solo sobre activos
    df_act = panel_modelo.filter(sf.col(cap) > 0)
    
    stats = df_act.select(
        sf.avg(cap).alias("media"),
        sf.stddev(cap).alias("desv_est"),
        sf.expr(f"percentile_approx(`{cap}`, 0.25)").alias("p25"),
        sf.expr(f"percentile_approx(`{cap}`, 0.50)").alias("p50"),
        sf.expr(f"percentile_approx(`{cap}`, 0.75)").alias("p75"),
    ).collect()[0]
    
    media = stats['media']
    cv = stats['desv_est'] / media if media and media > 0 else None
    iqr = stats['p75'] - stats['p25']
    
    cumple_cv = cv is not None and cv >= 0.30
    cumple_iqr = iqr >= 0.10 * media if media else False
    
    rec = 'Continua ' if (cumple_cv or cumple_iqr) else 'Binaria '
    
    print(f"\n{'='*45}")
    print(f"{cap}")
    print(f"  Nulls:                  {n_nulls:>10,}  ({n_nulls/n_total*100:.1f}%)")
    print(f"  Zeros (sin actividad):  {n_zeros:>10,}  ({n_zeros/n_total*100:.1f}%)")
    print(f"  Valores > 0:            {n_pos:>10,}  ({n_pos/n_total*100:.1f}%)")
    print(f"  Media:                  {media:>10.4f}")
    print(f"  Std:                    {stats['desv_est']:>10.4f}")
    print(f"  CV:                     {cv:>10.4f}  ({'CV ≥ 0.3 ' if cumple_cv else 'CV < 0.3 '})")
    print(f"  IQR:                    {iqr:>10.4f}  ({'IQR ≥ 10% media ' if cumple_iqr else 'IQR < 10% media '})")
    print(f"  P25/P50/P75:            {stats['p25']:.4f} / {stats['p50']:.4f} / {stats['p75']:.4f}")
    print(f"  Recomendación:          {rec}")
    
    resumen_c5.append({
        'capacidad': cap,
        'n_activos': n_pos,
        'pct_activos': round(n_pos/n_total*100, 1),
        'cv': round(cv, 4) if cv else None,
        'iqr_rel': round(iqr/media, 4) if media else None,
        'recomendacion': rec
    })

print("\nRESUMEN C5")
print(pd.DataFrame(resumen_c5).to_string(index=False))

In [ ]:
# Histogramas de distribución 
n_caps = len(CAPACIDADES_CONTINUAS)
fig, axes = plt.subplots(1, n_caps, figsize=(6 * n_caps, 5))
if n_caps == 1:
    axes = [axes]

fig.suptitle('C5: Distribución de uso por capacidad',
             fontsize=13, fontweight='bold', y=1.02)
colores_hist = ['#4A7DB5', '#5BAD8F', '#E8973A', '#E8593C', '#9B59B6']

for ax, cap, color in zip(axes, CAPACIDADES_CONTINUAS, colores_hist):
    if cap not in panel_modelo.columns:
        ax.set_visible(False)
        continue

    sample = (panel_modelo.filter(sf.col(cap) > 0).select(cap).sample(fraction=0.30, seed=42).toPandas())
    
    s = sample[cap].dropna()
    
    if len(s) == 0:
        ax.set_title(f'{cap} — sin datos')
        continue
    
    p25 = s.quantile(0.25)
    p50 = s.quantile(0.50)
    p75 = s.quantile(0.75)
    cv = s.std() / s.mean() if s.mean() > 0 else 0
    iqr = p75 - p25
    rec = 'Continua' if (cv >= 0.3 or iqr >= 0.1 * s.mean()) else 'Binaria'

    ax.hist(s, bins=40, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)

    counts, _ = np.histogram(s, bins=40)
    if counts.max() / max(counts[counts > 0].min(), 1) > 50:
        ax.set_yscale('log')
        ax.set_ylabel('Cantidad de obs (log)')
    else:
        ax.set_ylabel('Cantidad de obs')

    for p, lbl in [(p25, 'P25'), (p50, 'P50'), (p75, 'P75')]:
        ax.axvline(p, color='#333333', linewidth=1.2, linestyle='--', alpha=0.7)
        ax.text(p, ax.get_ylim()[1] * 0.5, lbl, ha='center', fontsize=9,
                color='#333333', fontweight='bold')

    textstr = f'CV: {cv:.2f}\nIQR: {iqr:.2f}\nn: {len(s):,}'
    ax.text(0.97, 0.95, textstr, transform=ax.transAxes, fontsize=9,
            va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))

    ax.set_title(f'{cap}', fontsize=11, fontweight='bold')
    ax.text(0.5, 1.01, f'Recomendación: {rec}',
            transform=ax.transAxes, ha='center', fontsize=10,
            color='#2C8A4A' if rec == 'Continua' else '#C0392B',
            fontweight='bold')
    ax.set_xlabel('Nivel de uso')
    ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Revenue core promedio por bin de uso
n_caps = len(CAPACIDADES_CONTINUAS)
fig, axes = plt.subplots(2, n_caps, figsize=(6 * n_caps, 10))
if n_caps == 1:
    axes = axes.reshape(2, 1)

fig.suptitle('C5: Revenue core promedio por bin de uso\n(solo obs con capacidad activa)',
             fontsize=13, fontweight='bold')

for i, cap in enumerate(CAPACIDADES_CONTINUAS):
    if cap not in panel_modelo.columns:
        continue

    max_val = panel_modelo.filter(sf.col(cap) > 0).agg(sf.max(cap)).collect()[0][0]
    
    if max_val <= 1.5:
        multiplicador = 100
        df_bin = (panel_modelo
            .filter(sf.col(cap) > 0)
            .filter(sf.col("ingreso_neto_core_real").isNotNull())
            .withColumn("bin",
                sf.when(sf.col(cap) * multiplicador > 100, 100)
                 .otherwise(sf.ceil(sf.col(cap) * multiplicador / 10) * 10)
                 .cast("int")
            )
            .withColumn("log_rev", sf.log(sf.col("ingreso_neto_core_real") + 1))
        )
        xlabel = 'Bin (%)'
    elif max_val <= 100:
        df_bin = (panel_modelo
            .filter(sf.col(cap) > 0)
            .filter(sf.col("ingreso_neto_core_real").isNotNull())
            .withColumn("bin",
                sf.when(sf.col(cap) > 100, 100)
                 .otherwise(sf.ceil(sf.col(cap) / 10) * 10)
                 .cast("int")
            )
            .withColumn("log_rev", sf.log(sf.col("ingreso_neto_core_real") + 1))
        )
        xlabel = 'Bin (%)'
    else:
        df_bin = (panel_modelo
            .filter(sf.col(cap) > 0)
            .filter(sf.col("ingreso_neto_core_real").isNotNull())
            .withColumn("bin",
                sf.when(sf.col(cap) >= 10, 10).otherwise(sf.col(cap)).cast("int")
            )
            .withColumn("log_rev", sf.log(sf.col("ingreso_neto_core_real") + 1))
        )
        xlabel = f'{cap} (valor, 10+ agrupado)'
    
    tabla = (df_bin
        .groupBy("bin")
        .agg(
            sf.count("*").alias("n"),
            sf.avg("ingreso_neto_core_real").alias("rev_avg"),
            sf.avg("log_rev").alias("log_avg"),
        )
        .filter(sf.col("n") >= 100)
        .orderBy("bin")
        .toPandas()
    )
    
    if len(tabla) == 0:
        continue
    
    axes[0, i].plot(tabla['bin'], tabla['rev_avg'], marker='o', color='steelblue', linewidth=2)
    axes[0, i].set_title(f'{cap}')
    axes[0, i].set_xlabel(xlabel)
    axes[0, i].set_ylabel('Revenue core promedio deflactado')
    axes[0, i].grid(True, alpha=0.3)
    
    axes[1, i].plot(tabla['bin'], tabla['log_avg'], marker='o', color='steelblue', linewidth=2)
    axes[1, i].set_title(f'{cap} — Log')
    axes[1, i].set_xlabel(xlabel)
    axes[1, i].set_ylabel('Log revenue core promedio')
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Binscatter residualizado (within PDV-período) 
# Residualización sobre el panel completo, esto replica exactamente la variación within que usa el TWFE para

if "period_id" not in panel_modelo.columns:
    min_periodo = panel_modelo.agg(sf.min("periodo")).collect()[0][0]
    panel_modelo = panel_modelo.withColumn(
        "period_id",
        ((sf.year("periodo") - sf.year(sf.lit(min_periodo))) * 12 +
         (sf.month("periodo") - sf.month(sf.lit(min_periodo)))).cast("int")
    )

n_caps = len(CAPACIDADES_CONTINUAS)
fig, axes = plt.subplots(1, n_caps, figsize=(6 * n_caps, 5))
if n_caps == 1:
    axes = [axes]

fig.suptitle('C5: Binscatter residualizado\n(within PDV-período sobre panel completo)',
             fontsize=13, fontweight='bold')

for ax, cap in zip(axes, CAPACIDADES_CONTINUAS):
    if cap not in panel_modelo.columns:
        ax.set_visible(False)
        continue

    # log_rev sobre TODO el panel — sin filtrar por cap > 0
    df_w = (panel_modelo
        .filter(sf.col("ingreso_neto_core_real").isNotNull())
        .withColumn("log_rev", sf.log(sf.col("ingreso_neto_core_real") + 1))
        .select("id_cliente", "period_id", cap, "log_rev")
    )
    
    # Medias sobre el panel completo
    medias = df_w.agg(
        sf.avg("log_rev").alias("y_global"),
        sf.avg(cap).alias("x_global")
    ).collect()[0]
    y_global = medias["y_global"]
    x_global = medias["x_global"]
    
    medias_pdv = df_w.groupBy("id_cliente").agg(
        sf.avg("log_rev").alias("y_pdv"),
        sf.avg(cap).alias("x_pdv")
    )
    
    medias_per = df_w.groupBy("period_id").agg(
        sf.avg("log_rev").alias("y_per"),
        sf.avg(cap).alias("x_per")
    )
    
    # Residualización sobre el panel completo
    df_resid = (df_w
        .join(medias_pdv, on="id_cliente", how="left")
        .join(medias_per, on="period_id", how="left")
        .withColumn("resid_y",
            sf.col("log_rev") - sf.col("y_pdv") - sf.col("y_per") + sf.lit(y_global))
        .withColumn("resid_x",
            sf.col(cap) - sf.col("x_pdv") - sf.col("x_per") + sf.lit(x_global))
        .select(cap, "resid_x", "resid_y")
    )
    
    n_total_resid = df_resid.count()
    sample_frac = min(1.0, 200000 / n_total_resid)
    sample = df_resid.sample(fraction=sample_frac, seed=42).toPandas()
    
    if len(sample) < 100:
        ax.set_title(f'{cap} — datos insuficientes')
        continue
    
    sample['bin'] = pd.qcut(sample['resid_x'], q=20, labels=False, duplicates='drop')
    binned = sample.groupby('bin')[['resid_x', 'resid_y']].mean()
    
    ax.scatter(binned['resid_x'], binned['resid_y'], color='steelblue', s=60)
    z = np.polyfit(binned['resid_x'], binned['resid_y'], 1)
    p = np.poly1d(z)
    x_sorted = sorted(binned['resid_x'])
    ax.plot(x_sorted, p(x_sorted), color='red', linestyle='--', alpha=0.7,
            label=f'Pendiente: {z[0]:.4f}')
    ax.axhline(0, color='black', linestyle='-', alpha=0.2)
    ax.axvline(0, color='black', linestyle='-', alpha=0.2)
    
    pct_activos = (sample[cap] > 0).mean() * 100
    ax.set_title(f'{cap}\n(n_sample: {len(sample):,}  |  {pct_activos:.1f}% activos)')
    ax.set_xlabel(f'Residuo {cap} (within PDV-período)')
    ax.set_ylabel('Residuo Log Revenue core (within PDV-período)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## C6. Criterio para definición de Historia de Datos 

In [ ]:
# Continuas:
#   Validación 1 — historia desde primera actividad ≥ 12 meses
#   Validación 2 — ≥ 20% de PDVs ACTIVOS con variación within sustantiva

# Binarias:
#   ≥ 80% de PDVs tratados en cohortes con ≥12 pre + ≥3 post

UMBRAL_MESES_HISTORIA   = 12
UMBRAL_PCT_VARIACION    = 0.20
UMBRAL_PCT_PDVS_BINARIA = 0.80
MIN_PRE  = 12
MIN_POST = 3

RANGOS_VARIACION = {
    'Digital':        0.05,   # escala 0-1
    'Multicategory':  0.05,   # escala 0-1
    'PedidoSugerido': 5.0,    # escala 0-100
    'Coolers':        1.0,    # conteo de puertas
}

panel_inicio = panel_modelo.agg(sf.min("periodo")).collect()[0][0]
panel_fin    = panel_modelo.agg(sf.max("periodo")).collect()[0][0]
meses_panel  = ((panel_fin.year - panel_inicio.year) * 12 +
                (panel_fin.month - panel_inicio.month) + 1)

print(f"Panel: {panel_inicio} - {panel_fin} ({meses_panel} meses)")

resultados_c6 = []
capacidades_excluidas_por_c6 = []

print(f"CAPACIDADES CONTINUAS — historia ≥{UMBRAL_MESES_HISTORIA}m | variación ≥{UMBRAL_PCT_VARIACION*100:.0f}% PDVs activos")

# CAPACIDADES BINARIAS
for cap in CAPACIDADES_CONTINUAS:
    if cap not in panel_modelo.columns:
        print(f"\n  '{cap}' no encontrada — omitiendo.")
        continue

    activos = panel_modelo.filter(sf.col(cap) > 0)
    n_obs_activas = activos.count()
    if n_obs_activas == 0:
        print(f"\n{cap} — sin actividad en el panel")
        capacidades_excluidas_por_c6.append(cap)
        resultados_c6.append({
            'Capacidad': cap, 'Tipo': 'Continua',
            '1er mes act': '—', 'Meses historia': 0,
            'PDVs activos': 0, '% PDVs con var': 0.0,
            'Resultado': ' Sin actividad'
        })
        continue
    
    # Historia desde primera actividad ─
    primer_mes = activos.agg(sf.min("periodo")).collect()[0][0]
    meses_historia = ((panel_fin.year - primer_mes.year) * 12 + (panel_fin.month - primer_mes.month) + 1)
    cumple_historia = meses_historia >= UMBRAL_MESES_HISTORIA
    
    # % de PDVs ACTIVOS con variación within sustantiva 
    rango_min = RANGOS_VARIACION.get(cap, 0.05)
    pdvs_activos_ids = activos.select("id_cliente").distinct()
    
    pdv_rango = (panel_modelo.join(pdvs_activos_ids, on="id_cliente", how="inner").groupBy("id_cliente").agg(
            sf.max(cap).alias("max_pdv"),
            sf.min(cap).alias("min_pdv"),
            sf.count("*").alias("n_obs")
        )
        .filter(sf.col("n_obs") >= 2)
        .withColumn("rango", sf.col("max_pdv") - sf.col("min_pdv"))
        .withColumn("tiene_variacion", sf.col("rango") >= rango_min)
    )
    
    n_total_activos = pdv_rango.count()
    n_con_var = pdv_rango.filter(sf.col("tiene_variacion")).count()
    pct_var = n_con_var / n_total_activos if n_total_activos > 0 else 0
    cumple_variacion = pct_var >= UMBRAL_PCT_VARIACION
    
    # Resultado
    if cumple_historia and cumple_variacion:
        resultado = ' Cumple'
    elif not cumple_historia and not cumple_variacion:
        resultado = f' No cumple — historia ({meses_historia}m) y variación ({pct_var*100:.1f}%)'
        capacidades_excluidas_por_c6.append(cap)
    elif not cumple_historia:
        resultado = f' No cumple — historia ({meses_historia}m)'
        capacidades_excluidas_por_c6.append(cap)
    else:
        resultado = f' No cumple — variación ({pct_var*100:.1f}%)'
        capacidades_excluidas_por_c6.append(cap)
    
    print(f"\n{cap}")
    print(f"  Primera actividad:           {primer_mes}")
    print(f"  Meses de historia:           {meses_historia}  (umbral ≥{UMBRAL_MESES_HISTORIA})")
    print(f"  Rango mínimo de cambio:      {rango_min}")
    print(f"  PDVs activos (alguna vez):   {n_total_activos:,}")
    print(f"  PDVs con variación:          {n_con_var:,} de {n_total_activos:,}  ({pct_var*100:.1f}%)  (umbral ≥{UMBRAL_PCT_VARIACION*100:.0f}%)")
    print(f"  Resultado:                   {resultado}")
    
    resultados_c6.append({
        'Capacidad':       cap,
        'Tipo':            'Continua',
        '1er mes act':     str(primer_mes),
        'Meses historia':  meses_historia,
        'PDVs activos':    n_total_activos,
        '% PDVs con var':  round(pct_var * 100, 1),
        'Resultado':       resultado,
    })

# CAPACIDADES BINARIAS
print(f"CAPACIDADES BINARIAS — cohortes con ≥{MIN_PRE} pre + ≥{MIN_POST} post; capacidad cumple si ≥{UMBRAL_PCT_PDVS_BINARIA*100:.0f}% de PDVs OK")

for cap in CAPACIDADES_BINARIAS:
    if cap not in panel_modelo.columns:
        print(f"\n  '{cap}' no encontrada — omitiendo.")
        continue

    fecha_activacion = (panel_modelo.filter(sf.col(cap) == 1).groupBy("id_cliente").agg(sf.min("periodo").alias("fecha_activacion")))
    
    if fecha_activacion.count() == 0:
        print(f"\n{cap} — sin PDVs tratados")
        capacidades_excluidas_por_c6.append(cap)
        continue
    
    cohortes = (fecha_activacion.groupBy("fecha_activacion").count().orderBy("fecha_activacion").toPandas())
    
    print(f"\n{cap} — por cohorte:")
    print(f"  {'Cohorte':<12} {'PDVs':>8} {'Pre':>5} {'Post':>5}  Resultado")
    print(f"  {'-'*55}")
    
    pdvs_ok = 0
    pdvs_tot = 0
    for _, row in cohortes.iterrows():
        f = row['fecha_activacion']
        n = row['count']
        pre  = (f.year - panel_inicio.year) * 12 + (f.month - panel_inicio.month)
        post = (panel_fin.year - f.year) * 12 + (panel_fin.month - f.month)
        
        cumple_pre  = pre >= MIN_PRE
        cumple_post = post >= MIN_POST
        
        if cumple_pre and cumple_post:
            res = ' Cumple'
            pdvs_ok += n
        elif not cumple_pre:
            res = ' Pre insuficiente'
        else:
            res = '  Post insuficiente'
        
        pdvs_tot += n
        print(f"  {str(f):<12} {n:>8,} {pre:>5} {post:>5}  {res}")
    
    pct_pdvs_ok = pdvs_ok / pdvs_tot if pdvs_tot > 0 else 0
    cumple_capacidad = pct_pdvs_ok >= UMBRAL_PCT_PDVS_BINARIA
    
    if cumple_capacidad:
        res_cap = f' Cumple ({pct_pdvs_ok*100:.1f}% PDVs OK)'
    else:
        res_cap = f' No cumple ({pct_pdvs_ok*100:.1f}% PDVs OK, umbral ≥{UMBRAL_PCT_PDVS_BINARIA*100:.0f}%)'
        capacidades_excluidas_por_c6.append(cap)
    
    print(f"\n  PDVs en cohortes que cumplen: {pdvs_ok:,} de {pdvs_tot:,} ({pct_pdvs_ok*100:.1f}%)")
    print(f"  Resultado capacidad:          {res_cap}")
    
    resultados_c6.append({
        'Capacidad':       cap,
        'Tipo':            'Binaria',
        '1er mes act':     str(cohortes['fecha_activacion'].min()),
        'Meses historia':  meses_panel,
        'PDVs activos':    pdvs_tot,
        '% PDVs con var':  round(pct_pdvs_ok * 100, 1),
        'Resultado':       res_cap,
    })

# Resumen
print("\nResumen C6")
resumen_c6 = pd.DataFrame(resultados_c6)
print(resumen_c6.to_string(index=False))

if capacidades_excluidas_por_c6:
    print(f"  Capacidades excluidas por C6: {capacidades_excluidas_por_c6}")
    print(f"   No se incluirán en C8 (co-ocurrencia) ni en C9 (combinaciones).")
else:
    print(f"Todas las capacidades cumplen C6.")

In [ ]:
# Distribución de meses de historia por PDV, por capacidad.
print("HISTORIA POR PDV")

print("\nMeses totales por PDV (todas las capacidades continuas)")
meses_por_pdv = (panel_modelo.groupBy("id_cliente").agg(sf.count("*").alias("meses")))
dist_meses = meses_por_pdv.groupBy("meses").count().orderBy("meses").toPandas()
print(dist_meses.to_string(index=False))

percentiles = meses_por_pdv.approxQuantile("meses", [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99], 0.01)
print(f"\nPercentiles de meses/PDV:")
print(f"  P01:  {percentiles[0]:.0f}")
print(f"  P05:  {percentiles[1]:.0f}")
print(f"  P25:  {percentiles[2]:.0f}")
print(f"  P50:  {percentiles[3]:.0f}")
print(f"  P75:  {percentiles[4]:.0f}")
print(f"  P95:  {percentiles[5]:.0f}")
print(f"  P99:  {percentiles[6]:.0f}")

for cap in CAPACIDADES_BINARIAS:
    if cap not in panel_modelo.columns:
        continue
    
    print(f"\nPre/Post por PDV tratado — {cap}")
    
    fecha_activacion = (panel_modelo.filter(sf.col(cap) == 1).groupBy("id_cliente").agg(sf.min("periodo").alias("fecha_activacion")))
    
    # Por cada PDV tratado, contar meses pre y post en el panel_modelo
    pre_post_pdv = (panel_modelo.join(fecha_activacion, on="id_cliente", how="inner")
        .withColumn("es_pre", sf.when(sf.col("periodo") < sf.col("fecha_activacion"), 1).otherwise(0))
        .withColumn("es_post", sf.when(sf.col("periodo") >= sf.col("fecha_activacion"), 1).otherwise(0))
        .groupBy("id_cliente")
        .agg(
            sf.sum("es_pre").alias("meses_pre"),
            sf.sum("es_post").alias("meses_post"),
        )
    )
    
    # Categorizar PDVs si son suficientes
    resumen_pdv = (pre_post_pdv.withColumn("categoria",
            sf.when((sf.col("meses_pre") >= 12) & (sf.col("meses_post") >= 3), "OK (12+pre, 3+post)")
             .when(sf.col("meses_pre") < 12, "Pre insuficiente (<12)")
             .when(sf.col("meses_post") < 3, "Post insuficiente (<3)")
             .otherwise("Otro"))
        .groupBy("categoria")
        .count()
        .orderBy(sf.col("count").desc())
        .toPandas()
    )
    
    print(resumen_pdv.to_string(index=False))
    
    # Distribución de meses post
    print(f"\nDistribución de meses POST por PDV tratado:")
    dist_post = (pre_post_pdv
        .groupBy("meses_post")
        .count()
        .orderBy("meses_post")
        .toPandas()
    )
    print(dist_post.to_string(index=False))

In [ ]:
# Cobertura temporal por capacidad
caps_a_graficar = [r for r in resultados_c6 if r['1er mes act'] != '—']

if caps_a_graficar:
    fig, ax = plt.subplots(figsize=(14, max(4, 0.5 * len(caps_a_graficar) + 1.5)))

    for i, r in enumerate(caps_a_graficar):
        cap = r['Capacidad']
        primer = pd.to_datetime(r['1er mes act'])
        cumple = r['Resultado'].strip().startswith('Cumple') or '✓' in r['Resultado']

        ax.barh(i, (primer - pd.Timestamp(panel_inicio)).days,
                left=pd.Timestamp(panel_inicio),
                height=0.55, color='#C8C5BF', edgecolor='none')

        color_post = '#2C8A4A' if cumple else '#C0392B'
        ax.barh(i, (pd.Timestamp(panel_fin) - primer).days,
                left=primer, height=0.55,
                color=color_post, edgecolor='none', alpha=0.75)

        ax.plot(primer, i, marker='o', color='black', markersize=6, zorder=3)

    labels = [f"{r['Capacidad']}  ({r['Tipo'][0]})" for r in caps_a_graficar]
    ax.set_yticks(range(len(caps_a_graficar)))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    ax.axvline(pd.Timestamp(panel_inicio), color='gray', linestyle=':', linewidth=1)
    ax.axvline(pd.Timestamp(panel_fin), color='gray', linestyle=':', linewidth=1)

    legend_elements = [
        Patch(facecolor='#C8C5BF', label='Pre (sin actividad)'),
        Patch(facecolor='#2C8A4A', alpha=0.75, label='Post — cumple C6'),
        Patch(facecolor='#C0392B', alpha=0.75, label='Post — no cumple C6'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

    ax.set_title(f'C6: Cobertura temporal por capacidad ({panel_inicio} a {panel_fin})',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Período')
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()


## C8. Criterio para definición de Co-ocurrencia entre capacidades

In [ ]:
# Heatmap asimétrico
UMBRAL_COOCURRENCIA = 80  # % — alerta de restricción estructural

todas_caps = CAPACIDADES_CONTINUAS + CAPACIDADES_BINARIAS
todas_caps = [c for c in todas_caps if c in panel_modelo.columns]
n = len(todas_caps)

print(f"Capacidades incluidas en C8: {todas_caps}")

agg_expr = [sf.max(sf.when(sf.col(cap) > 0, 1).otherwise(0)).alias(cap) for cap in todas_caps]
pdvs_caps = panel_modelo.groupBy("id_cliente").agg(*agg_expr)
pdvs_caps = pdvs_caps.persist(StorageLevel.MEMORY_AND_DISK)
total_pdvs_c8 = pdvs_caps.count()

# Calcular matriz de co-ocurrencia
cooc_matrix = np.zeros((n, n))

for i, col_i in enumerate(todas_caps):
    n_i = pdvs_caps.filter(sf.col(col_i) > 0).count()
    if n_i == 0:
        continue
    for j, col_j in enumerate(todas_caps):
        if i == j:
            cooc_matrix[i, j] = 100.0
            continue
        n_ij = pdvs_caps.filter((sf.col(col_i) > 0) & (sf.col(col_j) > 0)).count()
        cooc_matrix[i, j] = n_ij / n_i * 100

# Correlación de Kendall residualizada por período
# Residualizar cada capacidad restando la media del período, para eliminar tendencias temporales compartidas
# Sample a pandas para Kendall (max 100K obs)
sample_frac = min(1.0, 100000 / panel_modelo.count())
df_resid = panel_modelo.select(["id_cliente", "period_id"] + todas_caps).sample(fraction=sample_frac, seed=42).toPandas()

# Residualizar por período
for col in todas_caps:
    df_resid[col] = df_resid.groupby("period_id")[col].transform(lambda x: x - x.mean())

kendall_matrix = np.zeros((n, n))
pval_matrix = np.zeros((n, n))

for i, col_i in enumerate(todas_caps):
    for j, col_j in enumerate(todas_caps):
        if i == j:
            kendall_matrix[i, j] = 1.0
            pval_matrix[i, j] = 0.0
            continue
        par = df_resid[[col_i, col_j]].dropna()
        if len(par) < 1000:
            kendall_matrix[i, j] = np.nan
            pval_matrix[i, j] = np.nan
            continue
        tau, pval = scistats.kendalltau(par[col_i], par[col_j])
        kendall_matrix[i, j] = tau
        pval_matrix[i, j] = pval

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('C8: Co-ocurrencia y correlación entre capacidades\n(co-ocurrencia a nivel PDV, correlación residualizada por período)',
             fontsize=13, fontweight='bold')

# Heatmap co-ocurrencia
sns.heatmap(cooc_matrix, annot=True, fmt='.1f', cmap='RdYlGn_r',
            xticklabels=todas_caps, yticklabels=todas_caps,
            ax=axes[0], vmin=0, vmax=100, linewidths=0.5, linecolor='white',
            annot_kws={'size': 11})
axes[0].set_title(
    f'Co-ocurrencia a nivel PDV (%)\n'
    f'(% de PDVs con cap. fila que también tienen cap. columna)\n'
    f'Borde rojo = restricción estructural (>={UMBRAL_COOCURRENCIA}%)'
)
for i in range(n):
    for j in range(n):
        if i != j and cooc_matrix[i, j] >= UMBRAL_COOCURRENCIA:
            axes[0].add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='red', lw=3))

# Heatmap Kendall residualizado
annot = np.empty((n, n), dtype=object)
for i in range(n):
    for j in range(n):
        if i == j:
            annot[i, j] = '1.00'
        elif np.isnan(kendall_matrix[i, j]):
            annot[i, j] = 'nan'
        else:
            stars = ('***' if pval_matrix[i, j] < 0.001 else
                     '**'  if pval_matrix[i, j] < 0.01  else
                     '*'   if pval_matrix[i, j] < 0.05  else '')
            annot[i, j] = f'{kendall_matrix[i, j]:.3f}\n{stars}'

sns.heatmap(kendall_matrix, annot=annot, fmt='', cmap='RdYlGn',
            xticklabels=todas_caps, yticklabels=todas_caps,
            ax=axes[1], vmin=-1, vmax=1, linewidths=0.5, linecolor='white',
            annot_kws={'size': 10})
axes[1].set_title('Correlación de Kendall τ (residualizada por período)\n*** p<0.001  ** p<0.01  * p<0.05')

plt.tight_layout()
plt.show()

# Resumen
print(f"\nResumen C8 — Co-ocurrencia >= {UMBRAL_COOCURRENCIA}% a nivel PDV:")
alerta = False
for i, nom_i in enumerate(todas_caps):
    for j, nom_j in enumerate(todas_caps):
        if i != j and cooc_matrix[i, j] >= UMBRAL_COOCURRENCIA:
            print(f"   {nom_i} → {nom_j}: {cooc_matrix[i,j]:.1f}%")
            alerta = True
if not alerta:
    print("  Ningún par supera el 80%")

print("\nResumen C8 — Kendall tau residualizado:")
for i, nom_i in enumerate(todas_caps):
    for j, nom_j in enumerate(todas_caps):
        if i < j:
            tau = kendall_matrix[i, j]
            pval = pval_matrix[i, j]
            if np.isnan(tau):
                print(f"  {nom_i} x {nom_j}: tau=nan — masa insuficiente")
                continue
            sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
            fuerza = ('Fuerte '   if abs(tau) >= 0.5 else
                      'Moderada ' if abs(tau) >= 0.3 else
                      'Débil ')
            print(f"  {nom_i} x {nom_j}: tau={tau:.3f} {sig}  {fuerza}")

## C9. Criterio para definición de Umbral mínimo de PDVs 

In [ ]:
# Combinaciones manuales definidas por los pasos previos, historia, co-ocurrecia, etc
COMBINACIONES_MODELO = {
    "Digital":                                          ["Digital"],
    "Digital x Multicategory":                          ["Digital", "Multicategory"],
    "Digital x Coolers":                                ["Digital", "Coolers"],
    "Digital x RTM":                                    ["Digital", "RTM"],
    "Digital x Multicategory x Coolers":                ["Digital", "Multicategory", "Coolers"],
    "Digital x Multicategory x RTM":                    ["Digital", "Multicategory", "RTM"],
    "Digital x Coolers x RTM":                          ["Digital", "Coolers", "RTM"],
    "Digital x Multicategory x Coolers x RTM":          ["Digital", "Multicategory", "Coolers", "RTM"],
}

In [ ]:
ALPHA = 0.05
POWER = 0.80
EFFECT_SIZE = 0.10
PCT_NEGOCIO = 0.005  # 0.5% de la base
MIN_STREAK_MINIMO = 6
MIN_STREAK_IDEAL = 12

z_alpha = norm.ppf(1 - ALPHA / 2)
z_beta = norm.ppf(POWER)
umbral_estadistico = int(2 * ((z_alpha + z_beta) / EFFECT_SIZE) ** 2)
total_pdvs_c9 = panel_modelo.select("id_cliente").distinct().count()
umbral_negocio = int(total_pdvs_c9 * PCT_NEGOCIO)

print(f"Total PDVs:              {total_pdvs_c9:,}")
print(f"Umbral estadístico:      {umbral_estadistico:,}  (preferente)")
print(f"Umbral de negocio (N):   {umbral_negocio:,}  ({PCT_NEGOCIO*100:.1f}% de la base)")
print(f"Streak mínimo (X):       {MIN_STREAK_MINIMO} períodos consecutivos")
print(f"Streak ideal:            {MIN_STREAK_IDEAL} períodos consecutivos")


COMBINACIONES_MODELO = {
    k: v for k, v in COMBINACIONES_MODELO.items()
    if all(c in panel_modelo.columns for c in v)
}

print(f"\nCombinaciones a evaluar: {len(COMBINACIONES_MODELO)}")

def max_streak_consecutivo(periodos, conteos, umbral):
    max_streak = 0
    streak_actual = 0
    periodo_anterior = None
    for periodo, conteo in zip(periodos, conteos):
        es_consecutivo = (periodo_anterior is not None and periodo == periodo_anterior + 1)
        supera_umbral = conteo >= umbral
        if supera_umbral and (periodo_anterior is None or es_consecutivo):
            streak_actual += 1
            max_streak = max(max_streak, streak_actual)
        elif supera_umbral and not es_consecutivo:
            streak_actual = 1
            max_streak = max(max_streak, streak_actual)
        else:
            streak_actual = 0
        periodo_anterior = periodo
    return max_streak

def clasificar_estado(streak_estad, streak_neg):
    """Estado según qué umbrales pasa la combinación."""
    pasa_estad_ideal = streak_estad >= MIN_STREAK_IDEAL
    pasa_estad_min   = streak_estad >= MIN_STREAK_MINIMO
    pasa_neg_ideal   = streak_neg >= MIN_STREAK_IDEAL
    pasa_neg_min     = streak_neg >= MIN_STREAK_MINIMO
    
    if pasa_estad_ideal:
        return 'Ideal (estadístico)'
    if pasa_estad_min:
        return 'Mínimo (estadístico)'
    if pasa_neg_ideal:
        return 'Ideal (solo negocio)'
    if pasa_neg_min:
        return 'Mínimo (solo negocio)'
    return 'No medible'

periodos_completos = (panel_modelo
    .select("period_id").distinct()
    .orderBy("period_id")
    .toPandas()['period_id'].tolist()
)

resultados = []
series_tiempo = {}

for nombre, caps in COMBINACIONES_MODELO.items():
    cond = sf.col(caps[0]) > 0
    for c in caps[1:]:
        cond = cond & (sf.col(c) > 0)
    
    pdvs_por_periodo = (panel_modelo
        .filter(cond)
        .groupBy("period_id")
        .agg(sf.countDistinct("id_cliente").alias("n_pdvs"))
        .orderBy("period_id")
        .toPandas()
    )
    
    pdvs_por_periodo = (pd.DataFrame({'period_id': periodos_completos})
        .merge(pdvs_por_periodo, on='period_id', how='left')
        .fillna({'n_pdvs': 0})
    )
    pdvs_por_periodo['n_pdvs'] = pdvs_por_periodo['n_pdvs'].astype(int)
    
    series_tiempo[nombre] = pdvs_por_periodo
    
    if pdvs_por_periodo['n_pdvs'].sum() == 0:
        resultados.append({
            'Combinacion':       nombre,
            'Streak (estad.)':   0,
            'Streak (negocio)':  0,
            'PDVs min':          0,
            'PDVs max':          0,
            'Estado':            'Sin PDVs',
        })
        continue
    
    streak_estad = max_streak_consecutivo(
        pdvs_por_periodo['period_id'].tolist(),
        pdvs_por_periodo['n_pdvs'].tolist(),
        umbral_estadistico
    )
    streak_neg = max_streak_consecutivo(
        pdvs_por_periodo['period_id'].tolist(),
        pdvs_por_periodo['n_pdvs'].tolist(),
        umbral_negocio
    )
    
    estado = clasificar_estado(streak_estad, streak_neg)
    
    resultados.append({
        'Combinacion':       nombre,
        'Streak (estad.)':   streak_estad,
        'Streak (negocio)':  streak_neg,
        'PDVs min':          int(pdvs_por_periodo['n_pdvs'].min()),
        'PDVs max':          int(pdvs_por_periodo['n_pdvs'].max()),
        'Estado':            estado,
    })

res_df = pd.DataFrame(resultados).sort_values('Streak (estad.)', ascending=False)
print()
print(res_df.to_string(index=False))

# Streak por combinación
mapa_colores = {
    'Ideal (estadístico)':   '#2C8A4A',
    'Mínimo (estadístico)':  '#5BAD8F',
    'Ideal (solo negocio)':  '#E8973A',
    'Mínimo (solo negocio)': '#E8593C',
    'No medible':            '#C0392B',
    'Sin PDVs':              '#C0392B',
}
color_estado = res_df['Estado'].map(mapa_colores)

fig, ax = plt.subplots(figsize=(14, max(5, len(res_df) * 0.5)))
fig.suptitle(f'C9: Streak máximo de períodos consecutivos\n(estadístico ≥ {umbral_estadistico:,} preferente  |  negocio ≥ {umbral_negocio:,})',
             fontsize=13, fontweight='bold')

ax.barh(res_df['Combinacion'], res_df['Streak (estad.)'], color=color_estado, alpha=0.85, edgecolor='white', label='Streak estadístico')
ax.axvline(MIN_STREAK_IDEAL, color='#2C5F8A', linewidth=1.5, linestyle='--', label=f'Ideal ({MIN_STREAK_IDEAL})')
ax.axvline(MIN_STREAK_MINIMO, color='#E8593C', linewidth=1.5, linestyle='--', label=f'Mínimo ({MIN_STREAK_MINIMO})')

for i, (_, row) in enumerate(res_df.iterrows()):
    ax.text(row['Streak (estad.)'] + 0.2, i,
            f"estad:{row['Streak (estad.)']}  neg:{row['Streak (negocio)']}  |  {row['Estado']}",
            va='center', fontsize=9)

ax.set_xlabel('Streak máximo (umbral estadístico)')
ax.legend(fontsize=9)
ax.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# PDVs simultáneos por período 
n_combos = len(COMBINACIONES_MODELO)
fig, axes = plt.subplots(n_combos, 1, figsize=(14, 2.5 * n_combos), sharex=False)
if n_combos == 1:
    axes = [axes]

fig.suptitle(f'C9: PDVs simultáneos por período\n(línea azul = estadístico: {umbral_estadistico:,}  |  línea roja = negocio: {umbral_negocio:,})',
             fontsize=13, fontweight='bold')

for ax, (nombre, serie) in zip(axes, series_tiempo.items()):
    if len(serie) == 0:
        ax.set_visible(False)
        continue
    row_res = res_df[res_df['Combinacion'] == nombre].iloc[0]
    colores_barra = ['#2C8A4A' if v >= umbral_estadistico else '#E8973A' if v >= umbral_negocio else '#C0392B' for v in serie['n_pdvs']]
    ax.bar(serie['period_id'], serie['n_pdvs'], color=colores_barra, alpha=0.8, edgecolor='white')
    ax.axhline(umbral_estadistico, color='#2C5F8A', linewidth=1.5, linestyle='--')
    ax.axhline(umbral_negocio, color='#E8593C', linewidth=1.5, linestyle='--')
    ax.set_title(f"{nombre}  —  {row_res['Estado']}", fontsize=10)
    ax.set_ylabel('PDVs simultáneos')
    ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nResumen C9:")
for estado in ['Ideal (estadístico)', 'Mínimo (estadístico)', 
               'Ideal (solo negocio)', 'Mínimo (solo negocio)',
               'No medible', 'Sin PDVs']:
    n = (res_df['Estado'] == estado).sum()
    if n > 0:
        print(f"  {estado}: {n}")

## C10. Criterio para definición de Tratamiento donde la mayoría de los PDVs tienen la capacidad activada y la variable de tratamiento en valores máximos

In [ ]:

UMBRAL_PENETRACION = 0.80
UMBRAL_EFECTO_TECHO = 0.50

VENTANA_RECIENTE_3 = 3
VENTANA_RECIENTE_6 = 6

print(f"Umbral penetración:   {UMBRAL_PENETRACION*100:.0f}% de PDVs sostenida")
print(f"Umbral efecto techo:  {UMBRAL_EFECTO_TECHO*100:.0f}% de obs en valor máximo")
print(f"Ventanas evaluadas:   últimos 3 meses, últimos 6 meses, máximo histórico")

todas_caps_c10 = CAPACIDADES_CONTINUAS + CAPACIDADES_BINARIAS
todas_caps_c10 = [c for c in todas_caps_c10 if c in panel_modelo.columns]

# Identificar últimos N períodos del panel
periodos_ordenados = (panel_modelo
    .select("periodo").distinct()
    .orderBy(sf.col("periodo").desc())
    .limit(VENTANA_RECIENTE_6)
    .toPandas()['periodo'].tolist()
)
ultimos_3 = sorted(periodos_ordenados[:VENTANA_RECIENTE_3])
ultimos_6 = sorted(periodos_ordenados[:VENTANA_RECIENTE_6])
print(f"\nÚltimos 3 meses: {ultimos_3[0]} → {ultimos_3[-1]}")
print(f"Últimos 6 meses: {ultimos_6[0]} → {ultimos_6[-1]}")

resultados_c10 = []
series_penetracion = {}

for cap in todas_caps_c10:
    if cap not in panel_modelo.columns:
        continue
    
    # PDVs activos por período
    pdvs_activos_por_per = (panel_modelo
        .filter(sf.col(cap) > 0)
        .groupBy("periodo")
        .agg(sf.countDistinct("id_cliente").alias("pdvs_activos"))
    )
    
    # PDVs totales por período
    pdvs_totales_por_per = (panel_modelo
        .groupBy("periodo")
        .agg(sf.countDistinct("id_cliente").alias("pdvs_totales"))
    )
    
    penetracion_per = (pdvs_totales_por_per
        .join(pdvs_activos_por_per, on="periodo", how="left")
        .withColumn("pdvs_activos", sf.coalesce(sf.col("pdvs_activos"), sf.lit(0)))
        .withColumn("pct_penetracion", sf.col("pdvs_activos") / sf.col("pdvs_totales") * 100)
        .orderBy("periodo")
        .toPandas()
    )
    
    series_penetracion[cap] = penetracion_per
    
    pct_max = penetracion_per['pct_penetracion'].max()
    periodo_max = penetracion_per.loc[penetracion_per['pct_penetracion'].idxmax(), 'periodo']
    
    pct_3m = penetracion_per[penetracion_per['periodo'].isin(ultimos_3)]['pct_penetracion'].mean()
    pct_6m = penetracion_per[penetracion_per['periodo'].isin(ultimos_6)]['pct_penetracion'].mean()
    
    # Efecto techo
    if cap in CAPACIDADES_CONTINUAS:
        activos = panel_modelo.filter(sf.col(cap) > 0)
        p90 = activos.approxQuantile(cap, [0.90], 0.001)[0]
        n_techo = activos.filter(sf.col(cap) >= p90).count()
        n_total_activos = activos.count()
        pct_techo = n_techo / n_total_activos * 100 if n_total_activos > 0 else 0
        descripcion_techo = f"% obs en P90+ ({p90:.4f})"
    else:
        obs_activas = panel_modelo.filter(sf.col(cap) == 1).count()
        obs_total = panel_modelo.count()
        pct_techo = obs_activas / obs_total * 100
        descripcion_techo = "% obs con valor = 1"
    
    alta_3m = pct_3m >= UMBRAL_PENETRACION * 100
    alta_6m = pct_6m >= UMBRAL_PENETRACION * 100
    alta_max = pct_max >= UMBRAL_PENETRACION * 100
    alta_penetracion = alta_3m or alta_6m or alta_max
    efecto_techo = pct_techo >= UMBRAL_EFECTO_TECHO * 100
    
    cuales = []
    if alta_3m: cuales.append("3m")
    if alta_6m: cuales.append("6m")
    if alta_max: cuales.append("max")
    detalle_alta = ",".join(cuales) if cuales else "ninguna"
    
    if alta_penetracion and efecto_techo:
        estado = 'Excluir como tratamiento — usar como covariable'
        nivel = 'critico'
    elif alta_penetracion:
        estado = f'Alta penetración ({detalle_alta}) sin efecto techo'
        nivel = 'medio'
    elif efecto_techo:
        estado = 'Efecto techo pero penetración baja'
        nivel = 'medio'
    else:
        estado = 'Sin alerta'
        nivel = 'ok'
    
    print(f"\n{cap}")
    print(f"  Penetración últimos 3m:   {pct_3m:.1f}%  {'(supera umbral)' if alta_3m else ''}")
    print(f"  Penetración últimos 6m:   {pct_6m:.1f}%  {'(supera umbral)' if alta_6m else ''}")
    print(f"  Penetración máxima:       {pct_max:.1f}%  (en {periodo_max}) {'(supera umbral)' if alta_max else ''}")
    print(f"  {descripcion_techo}: {pct_techo:.1f}%")
    print(f"  Estado:                   {estado}")
    
    resultados_c10.append({
        'Capacidad':       cap,
        'Tipo':            'Continua' if cap in CAPACIDADES_CONTINUAS else 'Binaria',
        '% pen. 3m':       round(pct_3m, 1),
        '% pen. 6m':       round(pct_6m, 1),
        '% pen. max':      round(pct_max, 1),
        'Período max':     str(periodo_max),
        '% en techo':      round(pct_techo, 1),
        'Estado':          estado,
        'Nivel':           nivel,
    })


res_c10_df = pd.DataFrame(resultados_c10)
print("\nRESUMEN C10")
print(res_c10_df.to_string(index=False))

# Penetración por período (línea por capacidad)
fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle('C10: Penetración por período (% PDVs con capacidad activa)',
             fontsize=13, fontweight='bold')

colores_cap = ['#4A7DB5', '#5BAD8F', '#E8973A', '#E8593C', '#9B59B6', '#34495E']

for (cap, serie), color in zip(series_penetracion.items(), colores_cap):
    ax.plot(serie['periodo'], serie['pct_penetracion'], marker='o', markersize=3,
            linewidth=1.5, color=color, label=cap, alpha=0.85)

ax.axhline(UMBRAL_PENETRACION * 100, color='#C0392B', linewidth=1.5, linestyle='--',
           label=f'Umbral ({UMBRAL_PENETRACION*100:.0f}%)')
ax.axvspan(ultimos_6[0], ultimos_6[-1], alpha=0.1, color='gray', label='Últimos 6 meses')
ax.set_xlabel('Período')
ax.set_ylabel('% PDVs con capacidad activa')
ax.set_ylim(0, 105)
ax.legend(fontsize=9, loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Barras comparativas (3m / 6m / max + techo)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('C10: Penetración (3m, 6m, máx) y efecto techo', fontsize=13, fontweight='bold')

x = np.arange(len(res_c10_df))
width = 0.25

axes[0].barh(x - width, res_c10_df['% pen. 3m'], width, label='Últimos 3m', color='#4A7DB5', alpha=0.85)
axes[0].barh(x,         res_c10_df['% pen. 6m'], width, label='Últimos 6m', color='#5BAD8F', alpha=0.85)
axes[0].barh(x + width, res_c10_df['% pen. max'], width, label='Máximo', color='#E8973A', alpha=0.85)
axes[0].axvline(UMBRAL_PENETRACION * 100, color='#C0392B', linewidth=1.5, linestyle='--',
                label=f'Umbral ({UMBRAL_PENETRACION*100:.0f}%)')
axes[0].set_yticks(x)
axes[0].set_yticklabels(res_c10_df['Capacidad'])
axes[0].set_xlabel('% PDVs con capacidad activa')
axes[0].set_title('Penetración por ventana')
axes[0].legend(fontsize=9, loc='lower right')
axes[0].grid(axis='x', linestyle='--', alpha=0.3)

color_map = {'critico': '#C0392B', 'medio': '#E8973A', 'ok': '#2C8A4A'}
color_alerta = res_c10_df['Nivel'].map(color_map)

axes[1].barh(res_c10_df['Capacidad'], res_c10_df['% en techo'], color=color_alerta, alpha=0.85, edgecolor='white')
axes[1].axvline(UMBRAL_EFECTO_TECHO * 100, color='#C0392B', linewidth=1.5, linestyle='--',
                label=f'Umbral ({UMBRAL_EFECTO_TECHO*100:.0f}%)')
for i, (_, row) in enumerate(res_c10_df.iterrows()):
    axes[1].text(row['% en techo'] + 0.5, i, f"{row['% en techo']:.1f}%", va='center', fontsize=9)
axes[1].set_xlabel('% obs en valor máximo')
axes[1].set_title('Efecto techo')
axes[1].legend(fontsize=9)
axes[1].grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nResumen C10:")
print(f"  Excluir como tratamiento: {(res_c10_df['Nivel'] == 'critico').sum()}")
print(f"  Alerta:                   {(res_c10_df['Nivel'] == 'medio').sum()}")
print(f"  Sin alerta:               {(res_c10_df['Nivel'] == 'ok').sum()}")

In [ ]:
# Cleanup final 
spark.catalog.clearCache()